In [92]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

# 데이터 로드 및 점검

In [2]:
df = pd.read_csv("./원본데이터/road_data_v1_final.csv")
df

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,DTCT_PD,ROAD_WIDTH,POTHOLE_SIZE,u,v,key,highway,dist_to_road,OSM_HIGHWAY,OSM_DIST,ROAD_LEVEL
0,1,36.308420,127.354424,대전광역시 서구 도안동 2088,도안동,도안동,2025-12-09 22:50,8,12.8,1298415193,1298414556,0,residential,1.640233,residential,1.640233,소로
1,2,36.308420,127.354424,대전광역시 서구 도안동 2088,도안동,도안동,2025-12-09 22:50,8,10.2,1298414556,1298415193,0,residential,1.640233,residential,1.640233,소로
2,3,36.309225,127.354056,대전광역시 서구 도안동 2088,도안동,도안동,2025-12-09 22:49,8,11.6,1298414556,1298415297,0,residential,3.648603,residential,3.648603,소로
3,4,36.309486,127.354003,대전광역시 서구 도안동 2088,도안동,도안동,2025-12-09 22:49,8,20.4,1298414556,1298415297,0,residential,1.389824,residential,1.389824,소로
4,5,36.309004,127.351702,대전광역시 서구 도안동 2090,도안동,도안동,2025-12-09 22:49,15,28.9,8262016755,951164297,0,tertiary,0.455712,tertiary,0.455712,중로
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,2187,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,2024-01-19 16:36,15,19.1,277760900,8242148843,0,secondary,10.982705,secondary,10.982705,중로
2187,2188,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,2024-01-19 16:36,15,26.4,8242148843,277760900,0,secondary,10.868036,secondary,10.868036,중로
2188,2189,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,2024-01-19 16:36,15,25.3,277760900,8242148843,0,secondary,10.868036,secondary,10.868036,중로
2189,2190,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,2024-01-19 16:26,25,26.7,462078787,3399376832,0,primary,2.302882,primary,2.302882,대로


## 데이터 점검

### 1

In [3]:
df.columns.tolist()

['DTCT_UID',
 'PLC_LTTD',
 'PLC_LGTD',
 'ROAD_INFO',
 'DSTA_LGLD_INFO',
 'DSTA_ADSTRD_INFO',
 'DTCT_PD',
 'ROAD_WIDTH',
 'POTHOLE_SIZE',
 'u',
 'v',
 'key',
 'highway',
 'dist_to_road',
 'OSM_HIGHWAY',
 'OSM_DIST',
 'ROAD_LEVEL']

In [4]:
df[["PLC_LTTD","PLC_LGTD","DTCT_PD"]].isna().sum()

PLC_LTTD    0
PLC_LGTD    0
DTCT_PD     0
dtype: int64

In [5]:
df["DTCT_PD"].min(), df["DTCT_PD"].max(), df.shape

('2024-01-19 16:26', '2025-12-09 22:50', (2191, 17))

### 2

In [6]:
# 이벤트 df (이미 로드되어 있다고 가정)
# df_event

gdf_event = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["PLC_LGTD"], df["PLC_LTTD"]),
    crs="EPSG:4326"
)

# 유성구 행정경계 (폴리곤)
gdf_yuseong = gpd.read_file("./원본데이터/yuseong_road_slim.geojson")  # 경로 수정
gdf_yuseong = gdf_yuseong.to_crs("EPSG:4326")

gdf_event["inside_yuseong"] = gdf_event.within(gdf_yuseong.unary_union)

print("Inside Yuseong:", gdf_event["inside_yuseong"].sum())
print("Outside Yuseong:", (~gdf_event["inside_yuseong"]).sum())
print("Total:", len(gdf_event))

C:\Users\hyebi\AppData\Local\Temp\ipykernel_39324\3561842409.py:14: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gdf_event["inside_yuseong"] = gdf_event.within(gdf_yuseong.unary_union)


Inside Yuseong: 0
Outside Yuseong: 2191
Total: 2191


### 3

In [7]:
gdf_blocks = gpd.read_file("./원본데이터/yuseong_road_integrated.gpkg")  # 레이어명 필요 시 layer=...
print("Number of blocks:", len(gdf_blocks))
print(gdf_blocks.columns.tolist())
block_id_candidates = [c for c in gdf_blocks.columns if "block" in c.lower() or "id" in c.lower()]
print("ID-like columns:", block_id_candidates)
print("Has block_id:", "block_id" in gdf_blocks.columns)

Number of blocks: 4430
['section_id', 'sub_id', 'length_m', 'ALWNC_DE', 'ALWNC_RESN', 'BSI_INT', 'ENG_RN', 'MVMN_DE', 'MVMN_RESN', 'MVM_RES_CD', 'NTFC_DE', 'OPERT_DE', 'RBP_CN', 'RDS_DPN_SE', 'RDS_MAN_NO', 'REP_CN', 'RN', 'RN_CD', 'ROAD_BT', 'ROAD_LT', 'ROA_CLS_SE', 'SIG_CD', 'WDR_RD_CD', 'road_id', 'road_length_m', 'section_order', 'geometry']
ID-like columns: ['section_id', 'sub_id', 'road_id']
Has block_id: False


### 4

In [8]:
gdf_blocks = gdf_blocks.to_crs("EPSG:5186")
gdf_blocks["centroid"] = gdf_blocks.geometry.centroid
gdf_blk_pt = gdf_blocks.set_geometry("centroid")
gdf_vds = gpd.read_file("./원본데이터/vds_info.json")  # EPSG 확인
gdf_vds = gdf_vds.to_crs("EPSG:5186")
gdf_match = gpd.sjoin_nearest(
    gdf_blk_pt,
    gdf_vds,
    how="left",
    distance_col="dist_m"
)
for r in [200, 300, 500, 1000]:
    covered = (gdf_match["dist_m"] <= r).sum()
    print(f"Radius {r}m coverage:", covered, "/", len(gdf_match),
          f"({covered/len(gdf_match)*100:.1f}%)")


DataSourceError: './원본데이터/vds_info.json' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.

### 5

In [9]:
gdf_event = gdf_event.to_crs("EPSG:5186")
gdf_event["DTCT_PD"] = pd.to_datetime(gdf_event["DTCT_PD"])

def normalize_events(gdf, radius_m=20, time_min=5):
    gdf = gdf.sort_values("DTCT_PD").copy()
    groups = []
    used = np.zeros(len(gdf), dtype=bool)

    for i, row in gdf.iterrows():
        if used[gdf.index.get_loc(i)]:
            continue

        t0 = row["DTCT_PD"]
        buf = row.geometry.buffer(radius_m)

        mask = (
            (~used) &
            (gdf.geometry.within(buf)) &
            (gdf["DTCT_PD"].between(t0 - pd.Timedelta(minutes=time_min),
                                     t0 + pd.Timedelta(minutes=time_min)))
        )

        idxs = np.where(mask)[0]
        used[idxs] = True
        groups.append(idxs)

    return len(groups)

print("Original events:", len(gdf_event))

for r in [10, 20, 30]:
    for t in [3, 5, 10]:
        n = normalize_events(gdf_event, radius_m=r, time_min=t)
        print(f"R={r}m, T=±{t}min → normalized events:", n)

Original events: 2191
R=10m, T=±3min → normalized events: 1598
R=10m, T=±5min → normalized events: 1597
R=10m, T=±10min → normalized events: 1597
R=20m, T=±3min → normalized events: 1535
R=20m, T=±5min → normalized events: 1533
R=20m, T=±10min → normalized events: 1530
R=30m, T=±3min → normalized events: 1497
R=30m, T=±5min → normalized events: 1495
R=30m, T=±10min → normalized events: 1491


### 6

In [10]:
gdf_event = gdf_event.to_crs("EPSG:5186")
gdf_blocks = gdf_blocks.to_crs("EPSG:5186")

gdf_evt_blk = gpd.sjoin_nearest(
    gdf_event,
    gdf_blocks,
    how="left",
    max_distance=50  # 기존에 사용하던 값
)

block_col = "block_id" if "block_id" in gdf_blocks.columns else gdf_blocks.index.name

cnt = gdf_evt_blk.groupby(["section_id", "sub_id"]).size()

print("Top 20 blocks by event count")
print(cnt.sort_values(ascending=False).head(20))

print("\nZero-event block ratio:")
zero_ratio = 1 - (cnt.count() / len(gdf_blocks))
print(f"{zero_ratio*100:.2f}%")

print("\nEvent count value counts:")
print(cnt.value_counts().sort_index().head(10))

Top 20 blocks by event count
section_id   sub_id       
3167022_003  3167022_003_2    82
3167083_001  3167083_001_8    49
3167043_002  3167043_002_4    42
3167022_011  3167022_011_1    32
3167083_001  3167083_001_6    29
3166054_001  3166054_001_1    28
3167043_002  3167043_002_1    26
2167001_001  2167001_001_8    24
             2167001_001_1    20
3167015_003  3167015_003_4    18
3167043_002  3167043_002_8    18
3167045_001  3167045_001_1    18
3167002_002  3167002_002_1    17
3167043_002  3167043_002_5    16
2167001_001  2167001_001_6    15
3167016_001  3167016_001_1    15
3167042_000  3167042_000_1    14
3167048_002  3167048_002_1    14
3167083_001  3167083_001_7    13
3167047_002  3167047_002_4    13
dtype: int64

Zero-event block ratio:
91.81%

Event count value counts:
1     112
2     112
3      34
4      21
5      17
6      21
7       9
8       5
9       5
10      1
Name: count, dtype: int64


In [11]:
print("Event bounds:", gdf_event.to_crs("EPSG:4326").total_bounds)
print("Yuseong bounds:", gdf_yuseong.to_crs("EPSG:4326").total_bounds)

gdf_event[["PLC_LTTD","PLC_LGTD"]].head(5)


Event bounds: [127.256012   36.1107586 128.3140924  36.6804626]
Yuseong bounds: [127.24685185  36.26425319 127.41661234  36.49746256]


,PLC_LTTD,PLC_LGTD
0,36.308420,127.354424
1,36.308420,127.354424
2,36.309225,127.354056
3,36.309486,127.354003
4,36.309004,127.351702


## 유성구 도로 정보와 데이터 재매칭

In [12]:
import geopandas as gpd
from shapely.geometry import Point

gdf_event = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["PLC_LGTD"], df["PLC_LTTD"]),
    crs="EPSG:4326"
)

gdf_yuseong = gdf_yuseong.to_crs("EPSG:4326")

gdf_event_yuseong = gpd.sjoin(
    gdf_event,
    gdf_yuseong[["geometry"]],
    how="inner",
    predicate="intersects"
)

print("Total events:", len(gdf_event))
print("Inside Yuseong:", len(gdf_event_yuseong))
print("Outside Yuseong:", len(gdf_event) - len(gdf_event_yuseong))

gdf_event_yuseong = gdf_event_yuseong.drop(columns=["index_right"], errors="ignore")

df_event_yuseong = gdf_event_yuseong.drop(columns="geometry").copy()

print(df_event_yuseong[["PLC_LTTD", "PLC_LGTD"]].head())

Total events: 2191
Inside Yuseong: 0
Outside Yuseong: 2191
Empty DataFrame
Columns: [PLC_LTTD, PLC_LGTD]
Index: []


In [13]:
import geopandas as gpd

gdf_event = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["PLC_LGTD"], df["PLC_LTTD"]),
    crs="EPSG:4326"
).to_crs("EPSG:5186")

gdf_road = gpd.read_file("./원본데이터/yuseong_road_slim.geojson").to_crs("EPSG:5186")

# 도로 반경 30m (권장: 20~50m 실험)
ROAD_BUFFER_M = 30
gdf_road["geometry"] = gdf_road.geometry.buffer(ROAD_BUFFER_M)

gdf_event_yuseong = gpd.sjoin(
    gdf_event,
    gdf_road[["geometry"]],
    how="inner",
    predicate="intersects"
)

print("Total events:", len(gdf_event))
print("Near Yuseong roads:", len(gdf_event_yuseong))
print("Filtered out:", len(gdf_event) - len(gdf_event_yuseong))

Total events: 2191
Near Yuseong roads: 2641
Filtered out: -450


In [15]:
# 1) 도로 버퍼 생성
ROAD_BUFFER_M = 30
gdf_road_buf = gdf_road.copy()
gdf_road_buf["geometry"] = gdf_road_buf.geometry.buffer(ROAD_BUFFER_M)

# 2) 모든 도로 버퍼를 하나의 geometry로 통합
road_union = gdf_road_buf.union_all()

# 3) 이벤트가 도로 영역과 겹치는지 여부
gdf_event["near_road"] = gdf_event.intersects(road_union)

# 4) 필터링
gdf_event_yuseong = gdf_event[gdf_event["near_road"]].copy()

print("Total events:", len(gdf_event))
print("Near Yuseong roads:", len(gdf_event_yuseong))
print("Filtered out:", len(gdf_event) - len(gdf_event_yuseong))

Total events: 2191
Near Yuseong roads: 1469
Filtered out: 722


# 데이터 전처리

## 데이터 정규화

In [16]:
import numpy as np
import pandas as pd

gdf_evt = gdf_event_yuseong.sort_values("DTCT_PD").copy()
gdf_evt["DTCT_PD"] = pd.to_datetime(gdf_evt["DTCT_PD"])

used = np.zeros(len(gdf_evt), dtype=bool)
groups = []

RADIUS_M = 20
TIME_MIN = 5

for i, row in gdf_evt.iterrows():
    idx_i = gdf_evt.index.get_loc(i)
    if used[idx_i]:
        continue

    t0 = row["DTCT_PD"]
    buf = row.geometry.buffer(RADIUS_M)

    mask = (
        (~used) &
        (gdf_evt.geometry.within(buf)) &
        (gdf_evt["DTCT_PD"].between(
            t0 - pd.Timedelta(minutes=TIME_MIN),
            t0 + pd.Timedelta(minutes=TIME_MIN)
        ))
    )

    idxs = np.where(mask)[0]
    used[idxs] = True
    groups.append(idxs)

print("Normalized events:", len(groups))

Normalized events: 993


In [17]:
rep_rows = []
for idxs in groups:
    rep_rows.append(gdf_evt.iloc[idxs[0]])

gdf_evt_norm = gpd.GeoDataFrame(rep_rows, crs=gdf_evt.crs).reset_index(drop=True)

print("Normalized representative events:", len(gdf_evt_norm))  # 993가 나와야 정상

Normalized representative events: 993


## 블럭 ID 생성

In [18]:
gdf_blocks = gdf_blocks.to_crs("EPSG:5186").copy()
gdf_blocks["block_id"] = gdf_blocks["section_id"].astype(str) + "_" + gdf_blocks["sub_id"].astype(str)

In [19]:
gdf_evt_norm = gdf_evt_norm.to_crs("EPSG:5186").copy()

gdf_evt_blk = gpd.sjoin_nearest(
    gdf_evt_norm,
    gdf_blocks[["block_id", "geometry"]],
    how="left",
    distance_col="dist_blk_m",
    max_distance=50  # 필요 시 30/50/70 비교
)

print("Matched events:", gdf_evt_blk["block_id"].notna().sum(), "/", len(gdf_evt_blk))
print("Unmatched events:", gdf_evt_blk["block_id"].isna().sum())
print("Distance stats (m):")
print(gdf_evt_blk["dist_blk_m"].describe())

Matched events: 992 / 993
Unmatched events: 1
Distance stats (m):
count    992.000000
mean       7.159800
std        6.868729
min        0.007507
25%        2.408282
50%        5.362493
75%        9.593876
max       47.849612
Name: dist_blk_m, dtype: float64


In [20]:
cnt = gdf_evt_blk.dropna(subset=["block_id"]).groupby("block_id").size()

print("Blocks with >=1 event:", cnt.size)
print("Total blocks:", len(gdf_blocks))

zero_ratio = 1 - (cnt.size / len(gdf_blocks))
print(f"Zero-event block ratio: {zero_ratio*100:.2f}%")

print("\nTop 20 blocks by normalized event count:")
print(cnt.sort_values(ascending=False).head(20))

print("\nEvent count value_counts (1~10):")
print(cnt.value_counts().sort_index().head(10))

print("\nQuantiles of event count:")
print(cnt.quantile([0.50, 0.75, 0.90, 0.95, 0.99]))

Blocks with >=1 event: 359
Total blocks: 4430
Zero-event block ratio: 91.90%

Top 20 blocks by normalized event count:
block_id
3167022_003_3167022_003_2    46
3167083_001_3167083_001_8    34
3166054_001_3166054_001_1    27
3167043_002_3167043_002_4    25
3167043_002_3167043_002_1    20
2167001_001_2167001_001_8    19
2167001_001_2167001_001_1    19
3167022_011_3167022_011_1    17
3167083_001_3167083_001_6    16
2167001_001_2167001_001_6    14
3167033_004_3167033_004_1    12
3167045_001_3167045_001_1    12
3167002_002_3167002_002_1    11
3167043_002_3167043_002_5    11
3167016_001_3167016_001_1    11
3167043_002_3167043_002_8    10
3010029_000_3010029_000_1     9
2166002_000_2166002_000_3     9
3167050_009_3167050_009_2     9
3342028_001_3342028_001_2     9
dtype: int64

Event count value_counts (1~10):
1     212
2      51
3      26
4      26
5       7
6       6
7       4
8       6
9       5
10      1
Name: count, dtype: int64

Quantiles of event count:
0.50     1.0
0.75     3.0
0.90  

In [21]:
df_block = gdf_blocks[["block_id"]].copy()

df_block["event_cnt"] = df_block["block_id"].map(cnt).fillna(0).astype(int)

# 위험(2) 기준: 이벤트 수 상위 10% 컷 (데이터 분포 따라 0.90~0.95 조정)
t = int(np.ceil(df_block.loc[df_block["event_cnt"] > 0, "event_cnt"].quantile(0.90)))
t = max(t, 2)  # 너무 낮아지는 경우 방지

def make_grade(x):
    if x == 0:
        return 0
    if x <= t:
        return 1
    return 2

df_block["risk_grade"] = df_block["event_cnt"].apply(make_grade)

print("Chosen threshold T:", t)
print(df_block["risk_grade"].value_counts().sort_index())

Chosen threshold T: 6
risk_grade
0    4071
1     328
2      31
Name: count, dtype: int64


In [22]:
print("event_cnt value_counts (top 20):")
print(
    df_block["event_cnt"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("Blocks with event_cnt >= 1:", (df_block["event_cnt"] >= 1).sum())
print("Blocks with event_cnt >= 5:", (df_block["event_cnt"] >= 5).sum())
print("Blocks with event_cnt >= 6:", (df_block["event_cnt"] >= 6).sum())
print("Blocks with event_cnt >= 7:", (df_block["event_cnt"] >= 7).sum())

event_cnt value_counts (top 20):
event_cnt
0     4071
1      212
2       51
3       26
4       26
5        7
6        6
7        4
8        6
9        5
10       1
11       3
12       2
14       1
16       1
17       1
19       2
20       1
25       1
27       1
Name: count, dtype: int64
Blocks with event_cnt >= 1: 359
Blocks with event_cnt >= 5: 44
Blocks with event_cnt >= 6: 37
Blocks with event_cnt >= 7: 31


In [23]:
df_block["grade_v1"] = df_block["event_cnt"].apply(
    lambda x: 0 if x == 0 else (1 if x <= 6 else 2)
)

print("Grade v1 distribution (0 / 1~6 / 7+):")
print(df_block["grade_v1"].value_counts().sort_index())

Grade v1 distribution (0 / 1~6 / 7+):
grade_v1
0    4071
1     328
2      31
Name: count, dtype: int64


In [24]:
df_block["grade_v2"] = df_block["event_cnt"].apply(
    lambda x: 0 if x == 0 else (1 if x <= 4 else 2)
)

print("Grade v2 distribution (0 / 1~4 / 5+):")
print(df_block["grade_v2"].value_counts().sort_index())

Grade v2 distribution (0 / 1~4 / 5+):
grade_v2
0    4071
1     315
2      44
Name: count, dtype: int64


In [25]:
print(gdf_blocks.drop(columns="geometry").dtypes)

section_id         object
sub_id             object
length_m          float64
ALWNC_DE           object
ALWNC_RESN         object
BSI_INT            object
ENG_RN             object
MVMN_DE            object
MVMN_RESN          object
MVM_RES_CD         object
NTFC_DE            object
OPERT_DE           object
RBP_CN             object
RDS_DPN_SE         object
RDS_MAN_NO          int64
REP_CN             object
RN                 object
RN_CD              object
ROAD_BT           float64
ROAD_LT           float64
ROA_CLS_SE         object
SIG_CD             object
WDR_RD_CD          object
road_id            object
road_length_m     float64
section_order       int64
centroid         geometry
block_id           object
dtype: object


## 라벨값 정의

In [27]:
# df_block: block_id, event_cnt 가 이미 있는 상태라고 가정

df_block = df_block.copy()

df_block["risk_grade"] = df_block["event_cnt"].apply(
    lambda x: 0 if x == 0 else (1 if x <= 4 else 2)
)

print("v2 risk_grade distribution:")
print(df_block["risk_grade"].value_counts().sort_index())

v2 risk_grade distribution:
risk_grade
0    4071
1     315
2      44
Name: count, dtype: int64


In [28]:
# 1) 블럭 속성 테이블 (geometry 제외)
df_feat = gdf_blocks.drop(columns=["geometry", "centroid"], errors="ignore").copy()

# 2) v2 라벨 테이블
df_lab = df_block[["block_id", "event_cnt", "risk_grade"]].copy()

# 3) 병합
df_train = df_feat.merge(df_lab, on="block_id", how="left")

# 4) 결측 방지 (안전장치)
df_train["event_cnt"] = df_train["event_cnt"].fillna(0).astype(int)
df_train["risk_grade"] = df_train["risk_grade"].fillna(0).astype(int)

print("df_train shape:", df_train.shape)
print("risk_grade distribution:")
print(df_train["risk_grade"].value_counts().sort_index())

df_train shape: (4430, 29)
risk_grade distribution:
risk_grade
0    4071
1     315
2      44
Name: count, dtype: int64


In [29]:
drop_cols = [
    "block_id", "section_id", "sub_id", "road_id",
    "RDS_MAN_NO"
]

X = df_train.drop(
    columns=[c for c in drop_cols if c in df_train.columns] + ["event_cnt", "risk_grade"]
)
y = df_train["risk_grade"]

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts().sort_index())
print("\nFeature dtypes:")
print(X.dtypes.value_counts())

X shape: (4430, 22)
y distribution:
risk_grade
0    4071
1     315
2      44
Name: count, dtype: int64

Feature dtypes:
object     17
float64     4
int64       1
Name: count, dtype: int64


In [30]:
df_train

,section_id,sub_id,length_m,ALWNC_DE,ALWNC_RESN,BSI_INT,ENG_RN,MVMN_DE,MVMN_RESN,MVM_RES_CD,...,ROAD_LT,ROA_CLS_SE,SIG_CD,WDR_RD_CD,road_id,road_length_m,section_order,block_id,event_cnt,risk_grade
0,1000013_000,1000013_000_1,2312.283384,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,53970.000,1,30200,1,000000,2312.283384,0,1000013_000_1000013_000_1,0,0
1,1000013_001,1000013_001_1,4245.580726,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,53970.000,1,30200,1,000001,20056.471214,1,1000013_001_1000013_001_1,0,0
2,1000013_001,1000013_001_2,2628.181975,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,53970.000,1,30200,1,000001,20056.471214,1,1000013_001_1000013_001_2,0,0
3,1000013_001,1000013_001_3,639.143026,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,53970.000,1,30200,1,000001,20056.471214,1,1000013_001_1000013_001_3,0,0
4,1000013_001,1000013_001_4,708.815680,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,53970.000,1,30200,1,000001,20056.471214,1,1000013_001_1000013_001_4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4425,4862306_000,4862306_000_1,450.192994,20221230,대전대덕 공공지원민간임대주택지구 도로개설공사 준공,20,Yongseongbuk-ro,20230105,대전대덕 공공지원민간임대주택지구 도로개설공사 준공,01,...,413.108,4,30200,3,002206,450.192994,0,4862306_000_4862306_000_1,0,0
4426,4862703_000,4862703_000_1,119.673913,20230616,장대공공지구 개발에 따른 도로구간 신설,20,Worldcupnam-ro 7beon-gil,20230616,장대공공지구 개발에 따른 도로구간 신설,01,...,144.280,4,30200,3,002207,119.673913,0,4862703_000_4862703_000_1,0,0
4427,4862704_000,4862704_000_1,118.255972,20230616,장대공공주택지구 개발에 따른 도로구간신설,20,Worldcupnam-ro 8beon-gil,20230616,장대공공주택지구 개발에 따른 도로구간신설,01,...,103.928,4,30200,3,002208,118.255972,0,4862704_000_4862704_000_1,0,0
4428,4863275_000,4863275_000_1,290.002197,20231215,유성 광역복합환승센터 진입로 신설,20,Yuseong-daero 653beon-gil,20231215,유성 광역복합환승센터 진입로 신설,01,...,261.675,4,30200,3,002209,290.002197,0,4863275_000_4863275_000_1,0,0


## 기상데이터

### 기상데이터 라벨 정의

In [31]:
import pandas as pd
import numpy as np

# 0) 정규화 이벤트 매핑 결과에서 필요한 컬럼만 준비
evt = gdf_evt_blk.dropna(subset=["block_id"]).copy()
evt["DTCT_PD"] = pd.to_datetime(evt["DTCT_PD"])

# 1) 이벤트를 "월"로 변환 (월 시작일로 통일)
evt["event_month"] = (
    evt["DTCT_PD"]
    .dt.to_period("M")
    .dt.to_timestamp(how="start")
)

# 2) 블럭-월별 이벤트 카운트 집계
blk_month_cnt = (
    evt.groupby(["block_id", "event_month"])
       .size()
       .reset_index(name="event_cnt_month")
)

# 3) 패널의 기준 월(as_of_month) 범위 생성
min_m = evt["event_month"].min()
max_m = evt["event_month"].max()

months = pd.date_range(min_m, max_m, freq="MS")  # 월 시작일
blocks = df_block[["block_id"]].copy()           # 4430개 블럭

panel = blocks.assign(key=1).merge(
    pd.DataFrame({"as_of_month": months, "key": 1}),
    on="key",
    how="outer"
).drop(columns=["key"])

# 4) 다음달(label_month) 정의
panel["label_month"] = (panel["as_of_month"] + pd.offsets.MonthBegin(1))

# 5) 다음달 이벤트 카운트(next_event_cnt) 붙이기
panel = panel.merge(
    blk_month_cnt.rename(columns={"event_month": "label_month", "event_cnt_month": "next_event_cnt"}),
    on=["block_id", "label_month"],
    how="left"
)

panel["next_event_cnt"] = panel["next_event_cnt"].fillna(0).astype(int)

# 6) 다음달 위험등급 라벨 생성 (v2 컷 유지)
def grade_v2(x: int) -> int:
    if x == 0:
        return 0
    if x <= 4:
        return 1
    return 2

panel["next_risk_grade"] = panel["next_event_cnt"].apply(grade_v2)

print(panel.shape)
print(panel["next_risk_grade"].value_counts().sort_index())

(106320, 5)
next_risk_grade
0    105762
1       531
2        27
Name: count, dtype: int64


### ASOS 일자료 기상 데이터 수집, 정규화, 롤링 피처 생성 및 병합

In [32]:
import requests
import pandas as pd
from io import StringIO

authKey = "_FW6tOhdRS2VurToXQUtMw"
stn = 133  # 대전

# 1) 조회 기간: panel 기준(롤링 90일 + 여유)
start_date = (panel["as_of_month"].min() - pd.Timedelta(days=120)).strftime("%Y%m%d")
end_date   = (panel["as_of_month"].max()).strftime("%Y%m%d")

print("ASOS fetch range:", start_date, "~", end_date)

# 2) API 호출
url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd3.php"
params = {
    "tm1": start_date,
    "tm2": end_date,
    "stn": stn,
    "help": 0,
    "authKey": authKey
}

resp = requests.get(url, params=params, timeout=60)
resp.encoding = resp.apparent_encoding
raw = resp.text

# 3) 주석/빈줄 제거
lines = [l for l in raw.splitlines() if l and not l.startswith("#")]

if len(lines) == 0:
    raise ValueError("ASOS 응답에서 데이터 라인을 찾지 못했습니다. authKey/기간/관측소 확인 필요")

# 4) 공백 구분으로 읽기 (헤더 없음)
df_raw = pd.read_csv(
    StringIO("\n".join(lines)),
    sep=r"\s+",
    header=None,
    engine="python"
)

print("df_raw shape:", df_raw.shape)
print(df_raw.head(3))

ASOS fetch range: 20230903 ~ 20251201
df_raw shape: (821, 56)
         0    1    2     3   4    5     6   7    8     9   ...  46   47  48  \
0  20230903  133  1.5  1294  14  3.5    19  14  6.1    27  ...  -9 -9.0  -9   
1  20230904  133  1.2  1065  14  3.3   835   9  5.7  2345  ...  -9 -9.0  -9   
2  20230905  133  1.6  1366  11  3.8  1023   9  6.0     2  ...  -9 -9.0  -9   

    49  50    51    52    53    54    55  
0 -9.0  -9  27.6  27.4  26.9  23.5  19.1  
1 -9.0  -9  28.2  27.5  26.9  23.6  19.1  
2 -9.0  -9  28.4  27.6  26.9  23.6  19.1  

[3 rows x 56 columns]


In [33]:
print("n_cols:", df_raw.shape[1])

n_cols: 56


In [34]:
import numpy as np

weather_daily = pd.DataFrame({
    "date": pd.to_datetime(df_raw.iloc[:, 0], format="%Y%m%d", errors="coerce"),
    "ta_avg": pd.to_numeric(df_raw.iloc[:, 10], errors="coerce"),
    "ta_max": pd.to_numeric(df_raw.iloc[:, 11], errors="coerce"),
    "ta_min": pd.to_numeric(df_raw.iloc[:, 13], errors="coerce"),
    "rn_day": pd.to_numeric(df_raw.iloc[:, 38], errors="coerce"),
})

# -9 계열 결측 처리
for c in ["ta_avg", "ta_max", "ta_min", "rn_day"]:
    weather_daily.loc[weather_daily[c] <= -8.9, c] = np.nan

# 날짜 없는 행 제거 + 정렬
weather_daily = (
    weather_daily
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print(weather_daily.head(5))
print("Freeze days total:", (weather_daily["ta_min"] < 0).sum())
print(weather_daily[["rn_day","ta_min","ta_max","ta_avg"]].describe())

        date  ta_avg  ta_max  ta_min  rn_day
0 2023-09-03    27.0    32.5    22.3     NaN
1 2023-09-04    27.4    31.5    24.0     NaN
2 2023-09-05    26.9    31.2    23.8     NaN
3 2023-09-06    26.0    31.3    21.0     NaN
4 2023-09-07    25.0    30.1    19.8     NaN
Freeze days total: 156
           rn_day      ta_min      ta_max      ta_avg
count  363.000000  804.000000  821.000000  818.000000
mean     9.269972   10.587313   20.199878   14.839120
std     20.441176   10.051465   10.105608    9.997096
min      0.000000   -8.800000   -6.000000   -8.500000
25%      0.000000    1.400000   11.600000    5.900000
50%      1.100000   10.700000   21.700000   15.700000
75%      8.300000   19.550000   28.400000   23.475000
max    168.400000   27.400000   36.700000   31.600000


In [35]:
# 1) 파생 변수
weather_daily["is_rain"] = (weather_daily["rn_day"] > 0).astype(int)
weather_daily["is_freeze"] = (weather_daily["ta_min"] < 0).astype(int)

# 2) as_of_month 롤링 집계 함수
def make_weather_features(as_of_month, df):
    end = as_of_month - pd.Timedelta(days=1)

    w30 = df[(df["date"] <= end) & (df["date"] > end - pd.Timedelta(days=30))]
    w90 = df[(df["date"] <= end) & (df["date"] > end - pd.Timedelta(days=90))]

    return pd.Series({
        "rain_sum_30d": w30["rn_day"].sum(),
        "rain_days_30d": w30["is_rain"].sum(),
        "freeze_days_30d": w30["is_freeze"].sum(),
        "temp_range_30d": (w30["ta_max"].mean() - w30["ta_min"].mean()) if not w30.empty else 0,
        "rain_sum_90d": w90["rn_day"].sum(),
        "rain_days_90d": w90["is_rain"].sum(),
        "freeze_days_90d": w90["is_freeze"].sum(),
    })

# 3) 월 목록
as_of_months = (
    panel[["as_of_month"]]
    .drop_duplicates()
    .sort_values("as_of_month")
    .reset_index(drop=True)
)

# 4) 피처 생성
weather_monthly = as_of_months.apply(
    lambda r: make_weather_features(r["as_of_month"], weather_daily),
    axis=1
)
weather_monthly["as_of_month"] = as_of_months["as_of_month"].values

print(weather_monthly.head())

   rain_sum_30d  rain_days_30d  freeze_days_30d  temp_range_30d  rain_sum_90d  \
0         124.1           10.0             14.0        6.786667         176.6   
1          47.8           10.0             20.0        8.146923         212.2   
2          93.9           13.0             13.0        8.950000         265.8   
3          57.9            9.0              7.0       11.460000         199.6   
4          32.7            7.0              0.0       12.260000         184.5   

   rain_days_90d  freeze_days_90d as_of_month  
0           21.0             28.0  2024-01-01  
1           29.0             49.0  2024-02-01  
2           33.0             48.0  2024-03-01  
3           32.0             41.0  2024-04-01  
4           29.0             21.0  2024-05-01  


In [36]:
weather_cols = [
    "rain_sum_30d","rain_days_30d","freeze_days_30d","temp_range_30d",
    "rain_sum_90d","rain_days_90d","freeze_days_90d"
]

# 기존 컬럼 있으면 제거 후 재병합
panel = panel.drop(columns=[c for c in weather_cols if c in panel.columns], errors="ignore")

panel = panel.merge(weather_monthly, on="as_of_month", how="left")

print(panel.shape)
print(panel[weather_cols].head())
print(panel[weather_cols].describe())

(106320, 12)
   rain_sum_30d  rain_days_30d  freeze_days_30d  temp_range_30d  rain_sum_90d  \
0         124.1           10.0             14.0        6.786667         176.6   
1          47.8           10.0             20.0        8.146923         212.2   
2          93.9           13.0             13.0        8.950000         265.8   
3          57.9            9.0              7.0       11.460000         199.6   
4          32.7            7.0              0.0       12.260000         184.5   

   rain_days_90d  freeze_days_90d  
0           21.0             28.0  
1           29.0             49.0  
2           33.0             48.0  
3           32.0             41.0  
4           29.0             21.0  
        rain_sum_30d  rain_days_30d  freeze_days_30d  temp_range_30d  \
count  106320.000000  106320.000000    106320.000000   106320.000000   
mean      123.408333       9.500000         5.875000        9.846068   
std       122.680424       3.708117         8.671414        1.889470

In [37]:
print("n_cols:", df_raw.shape[1])
print(weather_daily.head(3))

n_cols: 56
        date  ta_avg  ta_max  ta_min  rn_day  is_rain  is_freeze
0 2023-09-03    27.0    32.5    22.3     NaN        0          0
1 2023-09-04    27.4    31.5    24.0     NaN        0          0
2 2023-09-05    26.9    31.2    23.8     NaN        0          0


In [38]:
weather_cols = [
    "rain_sum_30d",
    "rain_days_30d",
    "freeze_days_30d",
    "temp_range_30d",
    "rain_sum_90d",
    "rain_days_90d",
    "freeze_days_90d",
]

print("Weather columns existence:")
for c in weather_cols:
    print(f"{c}: {'YES' if c in panel.columns else 'NO'}")

Weather columns existence:
rain_sum_30d: YES
rain_days_30d: YES
freeze_days_30d: YES
temp_range_30d: YES
rain_sum_90d: YES
rain_days_90d: YES
freeze_days_90d: YES


In [39]:
import os
from datetime import datetime

out_dir = "./전처리데이터"
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")
fname = f"panel_block_month_nextRisk_v2_weatherRoll30d90d_{ts}.csv"
out_path = os.path.join(out_dir, fname)

panel.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)
print("Rows, Cols:", panel.shape)

Saved: ./전처리데이터\panel_block_month_nextRisk_v2_weatherRoll30d90d_20251217_1009.csv
Rows, Cols: (106320, 12)


#### gdf_blocks 백업 저장

In [40]:
print(gdf_blocks.dtypes)

section_id         object
sub_id             object
length_m          float64
ALWNC_DE           object
ALWNC_RESN         object
BSI_INT            object
ENG_RN             object
MVMN_DE            object
MVMN_RESN          object
MVM_RES_CD         object
NTFC_DE            object
OPERT_DE           object
RBP_CN             object
RDS_DPN_SE         object
RDS_MAN_NO          int64
REP_CN             object
RN                 object
RN_CD              object
ROAD_BT           float64
ROAD_LT           float64
ROA_CLS_SE         object
SIG_CD             object
WDR_RD_CD          object
road_id            object
road_length_m     float64
section_order       int64
geometry         geometry
centroid         geometry
block_id           object
dtype: object


In [42]:
import os
from datetime import datetime

out_dir = "./전처리데이터"
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")

gdf_blocks_save = gdf_blocks.drop(columns=["centroid"], errors="ignore")

gdf_blocks_path = f"{out_dir}/gdf_blocks_10m_base_{ts}.gpkg"
gdf_blocks_save.to_file(
    gdf_blocks_path,
    layer="blocks",
    driver="GPKG"
)

print("Saved gdf_blocks (gpkg):", gdf_blocks_path)


Saved gdf_blocks (gpkg): ./전처리데이터/gdf_blocks_10m_base_20251217_1010.gpkg


In [43]:
gdf_blocks.drop(columns="geometry").to_csv(
    f"{out_dir}/gdf_blocks_10m_base_attr_{ts}.csv",
    index=False,
    encoding="utf-8-sig"
)

## 도로 파손 탐지 데이터와 블럭 단위 공간 정보 병합

### 블럭 단위 공간 정보 데이터 점검

In [44]:
road = pd.read_csv(
    "./원본데이터/road_data_v1_final.csv",
    encoding="utf-8-sig"
)

print("road rows:", len(road))
print("road columns:")
print(road.columns.tolist())

road rows: 2191
road columns:
['DTCT_UID', 'PLC_LTTD', 'PLC_LGTD', 'ROAD_INFO', 'DSTA_LGLD_INFO', 'DSTA_ADSTRD_INFO', 'DTCT_PD', 'ROAD_WIDTH', 'POTHOLE_SIZE', 'u', 'v', 'key', 'highway', 'dist_to_road', 'OSM_HIGHWAY', 'OSM_DIST', 'ROAD_LEVEL']


In [45]:
import geopandas as gpd

gdf_road_evt = gpd.GeoDataFrame(
    road,
    geometry=gpd.points_from_xy(road["PLC_LGTD"], road["PLC_LTTD"]),
    crs="EPSG:4326"
).to_crs(gdf_blocks.crs)

In [46]:
road_cols = [
    "ROAD_LEVEL",
    "ROAD_WIDTH",
    "POTHOLE_SIZE",
    "OSM_HIGHWAY"
]

gdf_road_evt = gdf_road_evt[road_cols + ["geometry"]]

gdf_evt_blk_attr = gpd.sjoin_nearest(
    gdf_road_evt,
    gdf_blocks[["block_id", "geometry"]],
    how="left",
    max_distance=50,          # 이미 검증된 값
    distance_col="dist_m"
)

block_road_attr = (
    gdf_evt_blk_attr
    .dropna(subset=["block_id"])
    .groupby("block_id")
    .agg({
        "ROAD_WIDTH": "median",
        "POTHOLE_SIZE": "median",
        "ROAD_LEVEL": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "OSM_HIGHWAY": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    })
    .reset_index()
)

In [47]:
gdf_blocks_road = gdf_blocks.merge(
    block_road_attr,
    on="block_id",
    how="left"
)

print(gdf_blocks_road[["ROAD_LEVEL", "ROAD_WIDTH"]].isna().mean())
display(gdf_blocks_road.head())

ROAD_LEVEL    0.918059
ROAD_WIDTH    0.918059
dtype: float64


,section_id,sub_id,length_m,ALWNC_DE,ALWNC_RESN,BSI_INT,ENG_RN,MVMN_DE,MVMN_RESN,MVM_RES_CD,...,road_id,road_length_m,section_order,geometry,centroid,block_id,ROAD_WIDTH,POTHOLE_SIZE,ROAD_LEVEL,OSM_HIGHWAY
0,1000013_000,1000013_000_1,2312.283384,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,000000,2312.283384,0,"LINESTRING (226162.774 407529.569, 226170.02 4...",POINT (226806.713 408317.685),1000013_000_1000013_000_1,NaN,NaN,NaN,NaN
1,1000013_001,1000013_001_1,4245.580726,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,000001,20056.471214,1,"LINESTRING (228441.807 409793.608, 228459.451 ...",POINT (227983.37 411784.596),1000013_001_1000013_001_1,NaN,NaN,NaN,NaN
2,1000013_001,1000013_001_2,2628.181975,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,000001,20056.471214,1,"LINESTRING (227715.762 413793.014, 227716.121 ...",POINT (228200.767 415009.482),1000013_001_1000013_001_2,NaN,NaN,NaN,NaN
3,1000013_001,1000013_001_3,639.143026,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,000001,20056.471214,1,"LINESTRING (228776.135 416189.829, 228803.864 ...",POINT (228913.786 416478.251),1000013_001_1000013_001_3,NaN,NaN,NaN,NaN
4,1000013_001,1000013_001_4,708.815680,20101228,국토해양부 노선명을 이용하여 명명함,2000,Honamjiseon Expressway,20141030,직권수정(속성변경),99,...,000001,20056.471214,1,"LINESTRING (229043.715 416770.292, 229070.155 ...",POINT (229182.06 417096.721),1000013_001_1000013_001_4,NaN,NaN,NaN,NaN


In [48]:
print([c for c in gdf_blocks.columns if "ROAD" in c or "OSM" in c])

['ROAD_BT', 'ROAD_LT']


### OSM 도로망 확보

In [49]:
import osmnx as ox
import geopandas as gpd

# 1) 블럭을 WGS84로 변환 (OSM은 4326 기반)
gdf_blk_wgs = gdf_blocks.to_crs("EPSG:4326")

# 2) bbox 계산 (minx, miny, maxx, maxy)
minx, miny, maxx, maxy = gdf_blk_wgs.total_bounds
print("bbox:", minx, miny, maxx, maxy)

bbox: 127.24685185367984 36.26425319239877 127.41661582539247 36.4974625622878


In [52]:
import osmnx as ox

print("OSMnx version:", ox.__version__)

G2 = ox.graph_from_place(
    "Yuseong-gu, Daejeon, South Korea",
    network_type="drive",
    simplify=True
)

print("OSM graph loaded")
print(G2)

OSMnx version: 2.0.7
OSM graph loaded
MultiDiGraph with 3939 nodes and 10649 edges


In [53]:
gdf_edges2 = ox.graph_to_gdfs(G2, nodes=False, edges=True)

print("Edges shape:", gdf_edges2.shape)
print(gdf_edges2["highway"].value_counts().head(10))

Edges shape: (10649, 14)
highway
residential       7788
secondary         1409
tertiary           623
primary            258
secondary_link     178
unclassified       100
tertiary_link       67
living_street       64
motorway_link       36
primary_link        34
Name: count, dtype: int64


### OSM_HIGHWAY ↔ ROAD_LEVEL 데이터 기반 매핑

In [54]:
road['OSM_HIGHWAY'].unique()

array(['residential', 'tertiary', 'secondary', 'secondary_link',
       'primary', 'tertiary_link', 'primary_link', 'unclassified',
       'trunk', 'trunk_link', 'motorway_link', 'living_street',
       'motorway'], dtype=object)

In [55]:
road['ROAD_LEVEL'].unique()

array(['소로', '중로', '미분류', '대로'], dtype=object)

In [56]:
ct = pd.crosstab(
    road["OSM_HIGHWAY"],
    road["ROAD_LEVEL"]
)

ct

ROAD_LEVEL,대로,미분류,소로,중로
OSM_HIGHWAY,,,,
living_street,0,0,2,0
motorway,3,0,0,0
motorway_link,0,18,0,0
primary,162,5,0,0
primary_link,0,0,12,0
residential,0,77,691,0
secondary,0,15,0,925
secondary_link,0,0,42,0
tertiary,0,0,0,188


In [57]:
mapping = (
    road
    .dropna(subset=["OSM_HIGHWAY", "ROAD_LEVEL"])
    .groupby("OSM_HIGHWAY")["ROAD_LEVEL"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)

mapping

{'living_street': '소로',
 'motorway': '대로',
 'motorway_link': '미분류',
 'primary': '대로',
 'primary_link': '소로',
 'residential': '소로',
 'secondary': '중로',
 'secondary_link': '소로',
 'tertiary': '중로',
 'tertiary_link': '소로',
 'trunk': '미분류',
 'trunk_link': '소로',
 'unclassified': '소로'}

In [58]:
mapping_final = dict(mapping)  # 네 mapping 복사

override = {
    "primary_link": "대로",
    "secondary_link": "중로",
    "tertiary_link": "중로",
    "trunk": "대로",
    "trunk_link": "대로",
    "motorway_link": "대로",
}

mapping_final.update(override)

print("mapping_final:", mapping_final)

mapping_final: {'living_street': '소로', 'motorway': '대로', 'motorway_link': '대로', 'primary': '대로', 'primary_link': '대로', 'residential': '소로', 'secondary': '중로', 'secondary_link': '중로', 'tertiary': '중로', 'tertiary_link': '중로', 'trunk': '대로', 'trunk_link': '대로', 'unclassified': '소로'}


In [59]:
roadlevel_to_width = {
    "대로": 25.0,
    "중로": 15.0,
    "소로": 8.0,
    "미분류": 10.0,
}

In [60]:
import geopandas as gpd

gdf_blocks_5186 = gdf_blocks.to_crs(5186).copy()
gdf_edges2_5186 = gdf_edges2.to_crs(5186).copy()

# centroid가 geometry 컬럼으로 들어가 있으면 to_file 에러나므로 여기서는 centroid를 "임시"로만 씀
blk_pt = gdf_blocks_5186.set_geometry("centroid")

blk_osm = gpd.sjoin_nearest(
    blk_pt,
    gdf_edges2_5186[["highway","geometry"]],
    how="left",
    distance_col="dist_osm_m",
    max_distance=50
)

# geometry를 원래 블럭 라인으로 복귀
blk_osm = blk_osm.set_geometry("geometry")

In [63]:
import pandas as pd

def normalize_highway(x):
    if isinstance(x, list):
        return x[0]              # list → 첫 번째 값
    if pd.isna(x):
        return None              # NaN → None
    return x                     # str 그대로

blk_osm["OSM_HIGHWAY"] = blk_osm["OSM_HIGHWAY"].apply(normalize_highway)
blk_osm["OSM_HIGHWAY"].apply(type).value_counts()

OSM_HIGHWAY
<class 'str'>         5093
<class 'NoneType'>    1064
Name: count, dtype: int64

In [64]:
use_mapping = mapping_final  # 또는 mapping

blk_osm["ROAD_LEVEL"] = (
    blk_osm["OSM_HIGHWAY"]
    .map(use_mapping)
    .fillna("미분류")
)

blk_osm["ROAD_WIDTH"] = blk_osm["ROAD_LEVEL"].map(roadlevel_to_width)

In [65]:
blk_osm["ROAD_LEVEL"].value_counts()

ROAD_LEVEL
소로     4465
미분류    1064
중로      539
대로       89
Name: count, dtype: int64

In [66]:
blk_osm["ROAD_WIDTH"].isna().mean()

np.float64(0.0)

In [67]:
blk_osm[["OSM_HIGHWAY", "ROAD_LEVEL", "ROAD_WIDTH"]].sample(10)

,OSM_HIGHWAY,ROAD_LEVEL,ROAD_WIDTH
3312,residential,소로,8.0
3810,residential,소로,8.0
3407,residential,소로,8.0
4067,residential,소로,8.0
4126,residential,소로,8.0
736,residential,소로,8.0
3596,residential,소로,8.0
387,residential,소로,8.0
3805,residential,소로,8.0
4174,residential,소로,8.0


### 도로 파손 이벤트 데이터 block_id에 매핑

- 매핑 기준: block LineString 
- 매핑 방식: sjoin_nearest 
- max_distance: 30m (필요 시 50m) 
- CRS: EPSG:5186 (미터 단위)

#### 파손 이벤트 → GeoDataFrame 변환

In [71]:
road.shape

(2191, 17)

In [70]:
# GeoDataFrame 변환
gdf_evt = gpd.GeoDataFrame(
    road,
    geometry=gpd.points_from_xy(road["PLC_LGTD"], road["PLC_LTTD"]),
    crs="EPSG:4326"
).to_crs("EPSG:5186")

print("Event GDF:", gdf_evt.shape)

Event GDF: (2191, 18)


#### 블럭 GeoDataFrame CRS 정합 기준 설정

- 30m

In [72]:
gdf_blocks = gdf_blocks.to_crs("EPSG:5186")

print("Blocks GDF:", gdf_blocks.shape)

Blocks GDF: (4430, 29)


In [73]:
gdf_evt_blk_30 = gpd.sjoin_nearest(
    gdf_evt,
    gdf_blocks[["block_id", "geometry"]],
    how="left",
    distance_col="dist_to_block_m",
    max_distance=30
)

total_evt = len(gdf_evt_blk_30)
matched_30 = gdf_evt_blk_30["block_id"].notna().sum()
unmatched_30 = total_evt - matched_30

print("=== 30m 매핑 결과 ===")
print(f"총 이벤트 수        : {total_evt}")
print(f"매핑 성공 (30m)     : {matched_30}")
print(f"매핑 실패 (30m)     : {unmatched_30}")
print(f"성공률              : {matched_30 / total_evt:.2%}")

print("\n거리 통계 (m):")
print(gdf_evt_blk_30["dist_to_block_m"].describe())

=== 30m 매핑 결과 ===
총 이벤트 수        : 2191
매핑 성공 (30m)     : 1440
매핑 실패 (30m)     : 751
성공률              : 65.72%

거리 통계 (m):
count    1440.000000
mean        6.340480
std         5.360696
min         0.007507
25%         2.275687
50%         4.983810
75%         8.690236
max        29.801547
Name: dist_to_block_m, dtype: float64


In [74]:
gdf_evt_blk_30[gdf_evt_blk_30["block_id"].isna()][
    ["DTCT_UID", "PLC_LTTD", "PLC_LGTD"]
].head(10)

,DTCT_UID,PLC_LTTD,PLC_LGTD
0,1,36.308420,127.354424
1,2,36.308420,127.354424
2,3,36.309225,127.354056
3,4,36.309486,127.354003
4,5,36.309004,127.351702
5,6,36.309004,127.351702
6,7,36.309184,127.351612
7,8,36.312945,127.349898
8,9,36.317587,127.347850
9,10,36.317587,127.347850


- 50m

In [75]:
gdf_evt_blk_50 = gpd.sjoin_nearest(
    gdf_evt,
    gdf_blocks[["block_id", "geometry"]],
    how="left",
    distance_col="dist_to_block_m",
    max_distance=50
)

total_evt = len(gdf_evt_blk_50)
matched_50 = gdf_evt_blk_50["block_id"].notna().sum()
unmatched_50 = total_evt - matched_50

print("=== 50m 매핑 결과 ===")
print(f"총 이벤트 수        : {total_evt}")
print(f"매핑 성공 (50m)     : {matched_50}")
print(f"매핑 실패 (50m)     : {unmatched_50}")
print(f"성공률              : {matched_50 / total_evt:.2%}")

print("\n거리 통계 (m):")
print(gdf_evt_blk_50["dist_to_block_m"].describe())

=== 50m 매핑 결과 ===
총 이벤트 수        : 2191
매핑 성공 (50m)     : 1468
매핑 실패 (50m)     : 723
성공률              : 67.00%

거리 통계 (m):
count    1468.000000
mean        6.934419
std         6.850776
min         0.007507
25%         2.286868
50%         5.094781
75%         9.133834
max        47.849612
Name: dist_to_block_m, dtype: float64


##### 매핑 실패 원인 확인

In [ ]:
import geopandas as gpd

gdf_yuseong_road = gpd.read_file(
    "./원본데이터/yuseong_road_integrated.gpkg"
)
print(gdf_yuseong_road.shape)
print(gdf_yuseong_road.geometry.geom_type.value_counts())

gdf_evt_m = gdf_evt.to_crs("EPSG:5186")
gdf_yuseong_road_m = gdf_yuseong_road.to_crs("EPSG:5186")

COVER_BUFFER_M = 50

road_buf = gdf_yuseong_road_m[["geometry"]].copy()
road_buf["geometry"] = road_buf.geometry.buffer(COVER_BUFFER_M)

print("buffer polygons:", len(road_buf))

evt_in_road = gpd.sjoin(
    gdf_evt_m[["DTCT_UID", "geometry"]],
    road_buf,
    how="left",
    predicate="intersects"
)

evt_in_road["in_yuseong_road"] = evt_in_road.index_right.notna()

print("=== 전체 이벤트 기준 ===")
print("총 이벤트:", len(evt_in_road))
print("유성구 도로 안:", evt_in_road["in_yuseong_road"].sum())
print("유성구 도로 밖:", (~evt_in_road["in_yuseong_road"]).sum())

buffer polygons: 4430
=== 전체 이벤트 기준 ===
총 이벤트: 4663
유성구 도로 안: 3940
유성구 도로 밖: 723


In [80]:
# 50m 블럭 매핑 결과 기준
miss_mask = gdf_evt_blk_50["block_id"].isna()

miss_evt = evt_in_road.loc[miss_mask].copy()

print("\n=== 블럭 매핑 실패 이벤트 ===")
print("실패 이벤트 수:", len(miss_evt))
print("  유성구 도로 안:", miss_evt["in_yuseong_road"].sum())
print("  유성구 도로 밖:", (~miss_evt["in_yuseong_road"]).sum())


=== 블럭 매핑 실패 이벤트 ===
실패 이벤트 수: 723
  유성구 도로 안: 0
  유성구 도로 밖: 723


#### 블럭 GeoDataFrame CRS 정합

In [ ]:
# 1) 유성구 도로 안 도로 파손 이벤트만 유지
evt_in_scope = gdf_evt_blk_50[
    gdf_evt_blk_50["block_id"].notna()
].copy()

print("유성구 내 이벤트 수:", len(evt_in_scope))

유성구 내 이벤트 수: 1468


In [82]:
evt_in_scope

,DTCT_UID,PLC_LTTD,PLC_LGTD,ROAD_INFO,DSTA_LGLD_INFO,DSTA_ADSTRD_INFO,DTCT_PD,ROAD_WIDTH,POTHOLE_SIZE,u,...,key,highway,dist_to_road,OSM_HIGHWAY,OSM_DIST,ROAD_LEVEL,geometry,index_right,block_id,dist_to_block_m
17,18,36.332133,127.348003,대전광역시 서구 도안동 1485,도안동,도안동,2025-12-09 22:42,15,15.6,1391931734,...,0,secondary,1.179841,secondary,1.179841,중로,POINT (231245.213 414955.05),251.0,3166054_001_3166054_001_1,0.940947
18,19,36.332515,127.348037,대전광역시 유성구 원신흥동 667,원신흥동,원신흥동,2025-12-09 22:42,15,29.7,837542790,...,0,secondary,0.490203,secondary,0.490203,중로,POINT (231248.059 414997.428),251.0,3166054_001_3166054_001_1,9.141576
19,20,36.332492,127.348096,대전광역시 유성구 원신흥동 667,원신흥동,원신흥동,2025-12-09 22:42,15,29.6,837542790,...,0,secondary,6.368802,secondary,6.368802,중로,POINT (231253.419 414994.884),251.0,3166054_001_3166054_001_1,3.288981
20,21,36.335162,127.349085,대전광역시 유성구 원신흥동 667,원신흥동,원신흥동,2025-12-09 22:41,15,20.6,837542790,...,0,secondary,1.149820,secondary,1.149820,중로,POINT (231341.137 415291.494),251.0,3166054_001_3166054_001_1,3.539986
21,22,36.336635,127.349453,대전광역시 유성구 원신흥동 667,원신흥동,원신흥동,2025-12-09 22:41,15,24.5,837542790,...,0,secondary,1.525589,secondary,1.525589,중로,POINT (231373.541 415455.088),251.0,3166054_001_3166054_001_1,3.253775
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2186,2187,36.427741,127.387724,대전광역시 유성구 관평동 1286,관평동,관평동,2024-01-19 16:36,15,19.1,277760900,...,0,secondary,10.982705,secondary,10.982705,중로,POINT (234768.902 425577.888),1078.0,3167080_005_3167080_005_1,4.702843
2187,2188,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,2024-01-19 16:36,15,26.4,8242148843,...,0,secondary,10.868036,secondary,10.868036,중로,POINT (234732.917 425595.332),1078.0,3167080_005_3167080_005_1,4.660578
2188,2189,36.427899,127.387323,대전광역시 유성구 관평동 1286,관평동,관평동,2024-01-19 16:36,15,25.3,277760900,...,0,secondary,10.868036,secondary,10.868036,중로,POINT (234732.917 425595.332),1078.0,3167080_005_3167080_005_1,4.660578
2189,2190,36.398783,127.358005,대전광역시 유성구 장동 202-11,장동,신성동,2024-01-19 16:26,25,26.7,462078787,...,0,primary,2.302882,primary,2.302882,대로,POINT (232115.855 422354.186),93.0,2167001_001_2167001_001_8,1.425107


## block_id에 따른 월별 파손 이벤트 집계

### 이벤트 시간 컬럼 정리 (월 단위)

In [84]:
import pandas as pd

evt = evt_in_scope.copy()

# datetime 변환 (이미 되어있으면 자동 통과)
evt["DTCT_PD"] = pd.to_datetime(evt["DTCT_PD"])

# 이벤트 발생 월 (월 시작일 기준으로 통일)
evt["event_month"] = (
    evt["DTCT_PD"]
    .dt.to_period("M")
    .dt.to_timestamp(how="start")
)

print("이벤트 기간 범위:")
print(evt["event_month"].min(), "→", evt["event_month"].max())

이벤트 기간 범위:
2024-01-01 00:00:00 → 2025-12-01 00:00:00


In [85]:
# block × month 이벤트 카운트
blk_month_cnt = (
    evt.groupby(["block_id", "event_month"])
       .size()
       .reset_index(name="event_cnt")
)

In [86]:
blk_month_label = blk_month_cnt.rename(
    columns={
        "event_month": "label_month",
        "event_cnt": "next_event_cnt"
    }
)

print("blk_month_label shape:", blk_month_label.shape)
print(blk_month_label.head())

blk_month_label shape: (567, 3)
                    block_id label_month  next_event_cnt
0  1000013_001_1000013_001_5  2024-04-01               1
1  1000013_001_1000013_001_7  2024-03-01               2
2  1000015_000_1000015_000_1  2025-12-01               2
3  1000015_000_1000015_000_2  2025-01-01               3
4  1000015_001_1000015_001_1  2024-02-01               1


## block 데이터, month 파손 데이터, 기상 데이터 통합

In [90]:
panel_clean = panel.drop(
    columns=["next_event_cnt", "next_risk_grade"],
    errors="ignore"
)

panel_evt = panel_clean.merge(
    blk_month_label,
    on=["block_id", "label_month"],
    how="left"
)

panel_evt["next_event_cnt"] = panel_evt["next_event_cnt"].fillna(0).astype(int)

def risk_grade_v2(x: int) -> int:
    if x == 0:
        return 0
    if x <= 4:
        return 1
    return 2

panel_evt["next_risk_grade"] = panel_evt["next_event_cnt"].apply(risk_grade_v2)

print("panel_evt shape:", panel_evt.shape)

print("\nnext_event_cnt 분포:")
print(panel_evt["next_event_cnt"].value_counts().head(10))

print("\nnext_risk_grade 분포:")
print(panel_evt["next_risk_grade"].value_counts().sort_index())

print("\n라벨 존재 비율:",
      (panel_evt["next_event_cnt"] > 0).mean())

panel_evt shape: (106320, 12)

next_event_cnt 분포:
next_event_cnt
0     105756
1        254
2        176
3         42
4         33
5         20
6          8
7          7
12         6
14         3
Name: count, dtype: int64

next_risk_grade 분포:
next_risk_grade
0    105756
1       505
2        59
Name: count, dtype: int64

라벨 존재 비율: 0.005304740406320542


### 데이터 백업

In [91]:
import os
from datetime import datetime

out_dir = "./전처리데이터"
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")

# geometry 컬럼 1개만 유지
gdf_blocks_save = gdf_blocks.drop(columns=["centroid"], errors="ignore")

path_gpkg = f"{out_dir}/gdf_blocks_10m_with_osm_{ts}.gpkg"
gdf_blocks_save.to_file(
    path_gpkg,
    layer="blocks",
    driver="GPKG"
)

print("Saved blocks GPKG:", path_gpkg)

Saved blocks GPKG: ./전처리데이터/gdf_blocks_10m_with_osm_20251217_1115.gpkg


In [92]:
path_csv = f"{out_dir}/panel_block_month_weather_label_v2_{ts}.csv"

panel_evt.to_csv(
    path_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved panel CSV:", path_csv)
print("Shape:", panel_evt.shape)

Saved panel CSV: ./전처리데이터/panel_block_month_weather_label_v2_20251217_1115.csv
Shape: (106320, 12)


In [93]:
path_blocks_csv = f"{out_dir}/blocks_10m_with_osm_{ts}.csv"

gdf_blocks_save.drop(columns=["geometry"]).to_csv(
    path_blocks_csv,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Saved blocks CSV:", path_blocks_csv)

✅ Saved blocks CSV: ./전처리데이터/blocks_10m_with_osm_20251217_1115.csv


## 교통데이터

- 실시간 교통량을 불러올 수 있는 API밖에 없어서 보유 컬럼(gdf_blocks의 RN, RN_CD, section_id/sub_id, OSM 매핑된 OSM_HIGHWAY/ROAD_LEVEL/ROAD_WIDTH) 기준으로 “교차로 밀도/연결도/간선접근성” 같은 교통 프록시 피처 생성

- 공간 단위(행): block_id (10m 고정)

- 시간 단위(행): as_of_month (월 시작일)

- 라벨: next_risk_grade (다음달)

프록시 피처는 크게

- 정적 공간 피처(블럭 고정)
- 동적 시간 피처(월별 변동, 지금은 기상/라벨로 이미 있음)

으로 나뉘며 교통 프록시는 대부분 정적 공간 피처가 핵심

- 프록시 피처란, 우리가 “직접 측정하고 싶은 진짜 변수”를 대신해서 넣는 간접 지표
- 프로젝트에서 이상적인 진짜 변수는 이런 것
1. 블럭별 월평균 교통량 (AADT)
2. 중차량 비율
3. 누적 하중량
4. 차로별 반복 하중
5. 시간대별 통행 패턴
- 하지만 현실적으로
1. 과거 데이터 없음
2. 블럭 단위로 존재하지 않음
3. API도 실시간 위주

→ 직접 못 씀

- 그래서 생성한 cap_combo는 “이 블럭은 구조적으로 차량이 많이 다닐 수밖에 없는 도로인가?”를 수치로 표현한 것 = 교통량 프록시

### 0) CRS 통일 + 기본 확인

In [94]:
import geopandas as gpd
import pandas as pd
import numpy as np

# CRS 통일
gdf_blocks = gdf_blocks.to_crs("EPSG:5186")

gdf_edges2 = gdf_edges2.to_crs("EPSG:5186")
gdf_nodes2 = ox.graph_to_gdfs(G2, nodes=True, edges=False).to_crs("EPSG:5186")

print("Blocks:", gdf_blocks.shape)
print("Edges :", gdf_edges2.shape)
print("Nodes :", gdf_nodes2.shape)

Blocks: (4430, 29)
Edges : (10649, 14)
Nodes : (3939, 7)


### 1) 도로 등급 / 폭 기반 기본 처리용량 프록시

In [99]:
# gdf_blocks 대신 blk_osm을 메인으로 사용
gdf_blocks = blk_osm.copy()

print("Columns in gdf_blocks:")
print(gdf_blocks.columns.tolist())

# ROAD_LEVEL 존재 확인
print("ROAD_LEVEL exists:", "ROAD_LEVEL" in gdf_blocks.columns)
print("ROAD_WIDTH exists:", "ROAD_WIDTH" in gdf_blocks.columns)

Columns in gdf_blocks:
['section_id', 'sub_id', 'length_m', 'ALWNC_DE', 'ALWNC_RESN', 'BSI_INT', 'ENG_RN', 'MVMN_DE', 'MVMN_RESN', 'MVM_RES_CD', 'NTFC_DE', 'OPERT_DE', 'RBP_CN', 'RDS_DPN_SE', 'RDS_MAN_NO', 'REP_CN', 'RN', 'RN_CD', 'ROAD_BT', 'ROAD_LT', 'ROA_CLS_SE', 'SIG_CD', 'WDR_RD_CD', 'road_id', 'road_length_m', 'section_order', 'geometry', 'centroid', 'block_id', 'u', 'v', 'key', 'highway', 'dist_osm_m', 'OSM_HIGHWAY', 'ROAD_LEVEL', 'ROAD_WIDTH']
ROAD_LEVEL exists: True
ROAD_WIDTH exists: True


In [100]:
# ROAD_LEVEL → 수치화
roadlevel_score = {
    "소로": 1,
    "중로": 2,
    "대로": 3,
    "미분류": 1
}

gdf_blocks["cap_level_score"] = (
    gdf_blocks["ROAD_LEVEL"]
    .map(roadlevel_score)
    .fillna(1)
)

gdf_blocks["cap_width"] = gdf_blocks["ROAD_WIDTH"]

# 종합 처리용량 프록시
gdf_blocks["cap_combo"] = (
    gdf_blocks["cap_level_score"] * gdf_blocks["cap_width"]
)

print("=== cap_combo 통계 ===")
print(gdf_blocks["cap_combo"].describe())

=== cap_combo 통계 ===
count    6157.000000
mean       11.240052
std         9.872623
min         8.000000
25%         8.000000
50%         8.000000
75%        10.000000
max        75.000000
Name: cap_combo, dtype: float64


- 프록시 피처란, 우리가 “직접 측정하고 싶은 진짜 변수”를 대신해서 넣는 간접 지표
- 프로젝트에서 이상적인 진짜 변수는 이런 것
1. 블럭별 월평균 교통량 (AADT)
2. 중차량 비율
3. 누적 하중량
4. 차로별 반복 하중
5. 시간대별 통행 패턴
- 하지만 현실적으로
1. 과거 데이터 없음
2. 블럭 단위로 존재하지 않음
3. API도 실시간 위주

→ 직접 못 씀

- 그래서 생성한 cap_combo는 “이 블럭은 구조적으로 차량이 많이 다닐 수밖에 없는 도로인가?”를 수치로 표현한 것 = 교통량 프록시

### 2) 중차량(Heavy Vehicle) 하중 프록시

In [101]:
# -----------------------------
# 1) OSM 도로 유형 → 중차량 가중치
# -----------------------------
heavy_osm_weight = {
    "motorway": 3.0,
    "motorway_link": 2.5,
    "trunk": 2.5,
    "trunk_link": 2.0,
    "primary": 2.0,
    "primary_link": 1.7,
    "secondary": 1.5,
    "secondary_link": 1.3,
    "tertiary": 1.2,
    "tertiary_link": 1.1,
    "residential": 1.0,
    "living_street": 0.8,
    "unclassified": 1.0
}

In [102]:
# -----------------------------
# 2) 도로 등급 보정
# -----------------------------
roadlevel_heavy_weight = {
    "대로": 1.5,
    "중로": 1.2,
    "소로": 1.0,
    "미분류": 1.0
}

In [103]:
# -----------------------------
# 3) 중차량 프록시 계산
# -----------------------------
gdf_blocks["heavy_osm_score"] = (
    gdf_blocks["OSM_HIGHWAY"]
    .map(heavy_osm_weight)
    .fillna(1.0)
)

gdf_blocks["heavy_level_score"] = (
    gdf_blocks["ROAD_LEVEL"]
    .map(roadlevel_heavy_weight)
    .fillna(1.0)
)

# 최종 중차량 프록시
gdf_blocks["heavy_vehicle_proxy"] = (
    gdf_blocks["heavy_osm_score"] *
    gdf_blocks["heavy_level_score"]
)

print("=== heavy_vehicle_proxy 통계 ===")
print(gdf_blocks["heavy_vehicle_proxy"].describe())

print("\n=== ROAD_LEVEL별 평균 중차량 프록시 ===")
print(
    gdf_blocks
    .groupby("ROAD_LEVEL")["heavy_vehicle_proxy"]
    .mean()
    .sort_values(ascending=False)
)

print("\n=== OSM_HIGHWAY별 평균 중차량 프록시 (상위 10) ===")
print(
    gdf_blocks
    .groupby("OSM_HIGHWAY")["heavy_vehicle_proxy"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

print("\n=== 샘플 확인 ===")
display(
    gdf_blocks[
        ["OSM_HIGHWAY", "ROAD_LEVEL", "heavy_vehicle_proxy"]
    ].sample(10, random_state=42)
)

=== heavy_vehicle_proxy 통계 ===
count    6157.000000
mean        1.094229
std         0.356912
min         0.800000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.500000
Name: heavy_vehicle_proxy, dtype: float64

=== ROAD_LEVEL별 평균 중차량 프록시 ===
ROAD_LEVEL
대로     3.450000
중로     1.686679
미분류    1.000000
소로     0.998208
Name: heavy_vehicle_proxy, dtype: float64

=== OSM_HIGHWAY별 평균 중차량 프록시 (상위 10) ===
OSM_HIGHWAY
motorway          4.50
motorway_link     3.75
trunk             3.75
primary           3.00
trunk_link        3.00
primary_link      2.55
secondary         1.80
secondary_link    1.56
tertiary          1.44
residential       1.00
Name: heavy_vehicle_proxy, dtype: float64

=== 샘플 확인 ===


,OSM_HIGHWAY,ROAD_LEVEL,heavy_vehicle_proxy
3558,residential,소로,1.00
3405,residential,소로,1.00
2306,residential,소로,1.00
4071,residential,소로,1.00
1186,residential,소로,1.00
2369,residential,소로,1.00
1076,residential,소로,1.00
4187,residential,소로,1.00
2044,residential,소로,1.00
193,tertiary,중로,1.44


### 3) 교차로 밀도 (Intersection Density) 프록시

In [107]:
# -----------------------------
# 1) block buffer 생성
# -----------------------------
BUFFER_M = 30

gdf_blocks["geom_buffer_30m"] = gdf_blocks.geometry.buffer(BUFFER_M)

gdf_block_buf = gdf_blocks[["block_id", "geom_buffer_30m"]].copy()
gdf_block_buf = gdf_block_buf.set_geometry("geom_buffer_30m")

print("Block buffer 생성 완료")
print("Buffer 반경(m):", BUFFER_M)
print("Buffer GDF shape:", gdf_block_buf.shape)

Block buffer 생성 완료
Buffer 반경(m): 30
Buffer GDF shape: (6157, 2)


In [109]:
# -----------------------------
# 2) block buffer ↔ OSM nodes 공간조인
# -----------------------------
gdf_node_hit = gpd.sjoin(
    gdf_nodes2,
    gdf_block_buf,
    how="inner",
    predicate="within"
)

print("=== 공간조인 결과 ===")
print("매칭된 노드 수:", len(gdf_node_hit))
print("매칭된 block 수:", gdf_node_hit["block_id"].nunique())

=== 공간조인 결과 ===
매칭된 노드 수: 13731
매칭된 block 수: 3180


In [110]:
# -----------------------------
# 3) block별 교차로 개수 집계
# -----------------------------
intersection_cnt_30m = (
    gdf_node_hit
    .groupby("block_id")
    .size()
    .rename("intersection_cnt_30m")
    .reset_index()
)

print("=== 교차로 수 통계 (30m) ===")
print(intersection_cnt_30m["intersection_cnt_30m"].describe())

=== 교차로 수 통계 (30m) ===
count    3180.000000
mean        4.317925
std         3.999118
min         1.000000
25%         2.000000
50%         4.000000
75%         5.000000
max        78.000000
Name: intersection_cnt_30m, dtype: float64


In [111]:
# -----------------------------
# 4) gdf_blocks에 병합
# -----------------------------
gdf_blocks = gdf_blocks.merge(
    intersection_cnt_30m,
    on="block_id",
    how="left"
)

gdf_blocks["intersection_cnt_30m"] = gdf_blocks["intersection_cnt_30m"].fillna(0)

print("병합 후 gdf_blocks shape:", gdf_blocks.shape)

병합 후 gdf_blocks shape: (6157, 46)


### 4) 최종 traffic_load_proxy

In [113]:
# proxy 피처 생성용 GeoDataFrame
gdf_blocks_proxy = gdf_blocks.copy()

# 정규화 함수
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-6)

# 정규화
gdf_blocks_proxy["cap_norm"] = minmax(gdf_blocks_proxy["cap_combo"])
gdf_blocks_proxy["heavy_norm"] = minmax(gdf_blocks_proxy["heavy_vehicle_proxy"])
gdf_blocks_proxy["intersect_norm"] = minmax(gdf_blocks_proxy["intersection_cnt_30m"])

print("=== 정규화 확인 ===")
print(
    gdf_blocks_proxy[
        ["cap_norm", "heavy_norm", "intersect_norm"]
    ].describe()
)

=== 정규화 확인 ===
          cap_norm   heavy_norm  intersect_norm
count  6157.000000  6157.000000     6157.000000
mean      0.048359     0.079521        0.046133
std       0.147353     0.096463        0.051881
min       0.000000     0.000000        0.000000
25%       0.000000     0.054054        0.012821
50%       0.000000     0.054054        0.038462
75%       0.029851     0.054054        0.051282
max       1.000000     1.000000        1.000000


In [114]:
# 가중치 설정
W_CAP = 0.4
W_HEAVY = 0.4
W_INTERSECT = 0.2

gdf_blocks_proxy["traffic_load_proxy"] = (
    W_CAP * gdf_blocks_proxy["cap_norm"]
    + W_HEAVY * gdf_blocks_proxy["heavy_norm"]
    + W_INTERSECT * gdf_blocks_proxy["intersect_norm"]
)

print("=== traffic_load_proxy 통계 ===")
print(gdf_blocks_proxy["traffic_load_proxy"].describe())

print("\n=== 상위 10개 블록 ===")
display(
    gdf_blocks_proxy[
        ["block_id", "traffic_load_proxy", "cap_norm", "heavy_norm", "intersect_norm"]
    ]
    .sort_values("traffic_load_proxy", ascending=False)
    .head(10)
)

=== traffic_load_proxy 통계 ===
count    6157.000000
mean        0.060379
std         0.098120
min         0.002564
25%         0.026750
50%         0.031878
75%         0.037006
max         0.823077
Name: traffic_load_proxy, dtype: float64

=== 상위 10개 블록 ===


,block_id,traffic_load_proxy,cap_norm,heavy_norm,intersect_norm
7,1000013_001_1000013_001_7,0.823077,1.0,1.0,0.115385
5,1000013_001_1000013_001_5,0.815385,1.0,1.0,0.076923
6,1000013_001_1000013_001_6,0.815385,1.0,1.0,0.076923
2,1000013_001_1000013_001_2,0.810256,1.0,1.0,0.051282
1289,3167067_009_3167067_009_5,0.810256,1.0,1.0,0.051282
15,1000015_001_1000015_001_1,0.807692,1.0,1.0,0.038462
476,3167015_003_3167015_003_15,0.807692,1.0,1.0,0.038462
1270,3167067_003_3167067_003_1,0.805128,1.0,1.0,0.025641
411,3167010_002_3167010_002_1,0.802564,1.0,1.0,0.012821
3,1000013_001_1000013_001_3,0.802564,1.0,1.0,0.012821


### 5) 결과 데이터 백업

In [115]:
import os
from datetime import datetime

out_dir = "./전처리데이터"
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")

path_panel = f"{out_dir}/panel_block_month_weather_label_v1_{ts}.csv"

panel_evt.to_csv(
    path_panel,
    index=False,
    encoding="utf-8-sig"
)

print("Saved panel_evt:", path_panel)
print("Shape:", panel_evt.shape)

Saved panel_evt: ./전처리데이터/panel_block_month_weather_label_v1_20251217_1407.csv
Shape: (106320, 12)


In [118]:
from shapely import wkt

gdf_blocks_save = gdf_blocks_proxy.copy()

# geometry 컬럼 식별
geom_cols = gdf_blocks_save.select_dtypes(include="geometry").columns.tolist()
print("Geometry columns before save:", geom_cols)

# 대표 geometry 설정
gdf_blocks_save = gdf_blocks_save.set_geometry("geometry")

# 나머지 geometry → WKT
for c in geom_cols:
    if c != "geometry":
        gdf_blocks_save[c] = gdf_blocks_save[c].apply(
            lambda g: g.wkt if g is not None else None
        )

print("Geometry columns after fix:",
      gdf_blocks_save.select_dtypes(include="geometry").columns.tolist())

path_blocks = f"{out_dir}/gdf_blocks_proxy_traffic_v1_{ts}.gpkg"

gdf_blocks_save.to_file(
    path_blocks,
    layer="blocks_proxy",
    driver="GPKG"
)

print("Saved gdf_blocks_proxy:", path_blocks)
print("Shape:", gdf_blocks_save.shape)

Geometry columns before save: ['geometry', 'centroid', 'geom_buffer_30m']
Geometry columns after fix: ['geometry']
Saved gdf_blocks_proxy: ./전처리데이터/gdf_blocks_proxy_traffic_v1_20251217_1407.gpkg
Shape: (6157, 50)


In [120]:
path_evt_blk = f"{out_dir}/event_block_mapping_30m_v1_{ts}.csv"

gdf_evt_blk_30.drop(columns="geometry").to_csv(
    path_evt_blk,
    index=False,
    encoding="utf-8-sig"
)

print("Saved event-block mapping (30m):", path_evt_blk)
print("Shape:", gdf_evt_blk_30.shape)

Saved event-block mapping (30m): ./전처리데이터/event_block_mapping_30m_v1_20251217_1407.csv
Shape: (2191, 21)


## panel(기상 데이터)와 교통 proxy 결합

In [121]:
print("panel_evt shape:", panel_evt.shape)
print("gdf_blocks_proxy shape:", gdf_blocks_proxy.shape)

print("panel block_id unique:", panel_evt["block_id"].nunique())
print("proxy block_id unique:", gdf_blocks_proxy["block_id"].nunique())

# 결합 안 되는 block 확인
missing_blocks = (
    set(panel_evt["block_id"].unique())
    - set(gdf_blocks_proxy["block_id"].unique())
)
print("proxy에 없는 block 수:", len(missing_blocks))

panel_evt shape: (106320, 12)
gdf_blocks_proxy shape: (6157, 50)
panel block_id unique: 4430
proxy block_id unique: 4430
proxy에 없는 block 수: 0


In [122]:
blocks_proxy_tab = gdf_blocks_proxy.drop(columns="geometry")

print("blocks_proxy_tab shape:", blocks_proxy_tab.shape)
print("geometry in columns?", "geometry" in blocks_proxy_tab.columns)

# 병합 예정 컬럼 미리 확인
print(blocks_proxy_tab.columns.tolist())

blocks_proxy_tab shape: (6157, 49)
geometry in columns? False
['section_id', 'sub_id', 'length_m', 'ALWNC_DE', 'ALWNC_RESN', 'BSI_INT', 'ENG_RN', 'MVMN_DE', 'MVMN_RESN', 'MVM_RES_CD', 'NTFC_DE', 'OPERT_DE', 'RBP_CN', 'RDS_DPN_SE', 'RDS_MAN_NO', 'REP_CN', 'RN', 'RN_CD', 'ROAD_BT', 'ROAD_LT', 'ROA_CLS_SE', 'SIG_CD', 'WDR_RD_CD', 'road_id', 'road_length_m', 'section_order', 'centroid', 'block_id', 'u', 'v', 'key', 'highway', 'dist_osm_m', 'OSM_HIGHWAY', 'ROAD_LEVEL', 'ROAD_WIDTH', 'cap_level_score', 'cap_width', 'cap_combo', 'heavy_osm_score', 'heavy_level_score', 'heavy_vehicle_proxy', 'buffer_area', 'geom_buffer_30m', 'intersection_cnt_30m', 'cap_norm', 'heavy_norm', 'intersect_norm', 'traffic_load_proxy']


In [123]:
panel_full = panel_evt.merge(
    blocks_proxy_tab,
    on="block_id",
    how="left"
)

print("panel_full shape:", panel_full.shape)

# 핵심 컬럼 존재 확인
check_cols = [
    "traffic_load_proxy",
    "cap_combo",
    "heavy_vehicle_proxy",
    "intersection_cnt_30m",
    "ROAD_LEVEL",
    "ROAD_WIDTH"
]

for c in check_cols:
    print(f"{c} exists:", c in panel_full.columns)

# 결측 확인 (이론상 없어야 정상)
print("\n[결측률 확인]")
print(panel_full[check_cols].isna().mean())

panel_full shape: (147768, 60)
traffic_load_proxy exists: True
cap_combo exists: True
heavy_vehicle_proxy exists: True
intersection_cnt_30m exists: True
ROAD_LEVEL exists: True
ROAD_WIDTH exists: True

[결측률 확인]
traffic_load_proxy      0.0
cap_combo               0.0
heavy_vehicle_proxy     0.0
intersection_cnt_30m    0.0
ROAD_LEVEL              0.0
ROAD_WIDTH              0.0
dtype: float64


### row 수 늘어남 원인 확인 및 제거

In [124]:
dup = blocks_proxy_tab["block_id"].value_counts()
print("중복 block_id 개수:", (dup > 1).sum())
print("최대 중복 횟수:", dup.max())

print("\n중복 TOP 20:")
print(dup[dup > 1].head(20))

중복 block_id 개수: 1715
최대 중복 횟수: 6

중복 TOP 20:
block_id
3167003_004_3167003_004_1    6
3167006_002_3167006_002_1    6
4301168_001_4301168_001_3    6
4301047_001_4301047_001_2    2
4862704_000_4862704_000_1    2
4301051_002_4301051_002_2    2
4301051_003_4301051_003_1    2
4301040_000_4301040_000_2    2
4301053_002_4301053_002_1    2
4301049_000_4301049_000_1    2
4301049_001_4301049_001_1    2
4301050_000_4301050_000_1    2
4301043_001_4301043_001_1    2
4301043_001_4301043_001_2    2
4301043_002_4301043_002_1    2
4301044_000_4301044_000_1    2
4301044_000_4301044_000_2    2
4301047_000_4301047_000_3    2
4301047_000_4301047_000_4    2
4301047_001_4301047_001_1    2
Name: count, dtype: int64


In [125]:
import numpy as np
import pandas as pd

num_cols = blocks_proxy_tab.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [
    c for c in blocks_proxy_tab.columns
    if c not in num_cols + ["block_id"]
]

print("수치 컬럼 수:", len(num_cols))
print("범주 컬럼 수:", len(cat_cols))

수치 컬럼 수: 23
범주 컬럼 수: 25


In [126]:
def mode_or_first(s: pd.Series):
    s = s.dropna()
    if len(s) == 0:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]

agg_dict = {c: "mean" for c in num_cols}
agg_dict.update({c: mode_or_first for c in cat_cols})

blocks_proxy_1row = (
    blocks_proxy_tab
    .groupby("block_id", as_index=False)
    .agg(agg_dict)
)

print("축약 후 shape:", blocks_proxy_1row.shape)
print("축약 후 block_id unique:", blocks_proxy_1row["block_id"].nunique())

축약 후 shape: (4430, 49)
축약 후 block_id unique: 4430


In [127]:
panel_full = panel_evt.merge(
    blocks_proxy_1row,
    on="block_id",
    how="left"
)

print("panel_full shape:", panel_full.shape)

panel_full shape: (106320, 60)


In [128]:
check_cols = [
    "traffic_load_proxy",
    "cap_combo",
    "heavy_vehicle_proxy",
    "intersection_cnt_30m",
    "ROAD_LEVEL",
    "ROAD_WIDTH"
]

print(panel_full[check_cols].isna().mean())

traffic_load_proxy      0.0
cap_combo               0.0
heavy_vehicle_proxy     0.0
intersection_cnt_30m    0.0
ROAD_LEVEL              0.0
ROAD_WIDTH              0.0
dtype: float64


### 최종 데이터 파일 백업

In [129]:
import os
from datetime import datetime

out_dir = "./전처리데이터"
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")

path_panel = f"{out_dir}/panel_block_month_weather_proxy_label_v1_{ts}.csv"

panel_full.to_csv(
    path_panel,
    index=False,
    encoding="utf-8-sig"
)

print("=== panel_full 저장 완료 ===")
print("Path :", path_panel)
print("Shape:", panel_full.shape)
print("Columns:", len(panel_full.columns))

=== panel_full 저장 완료 ===
Path : ./전처리데이터/panel_block_month_weather_proxy_label_v1_20251217_1433.csv
Shape: (106320, 60)
Columns: 60


In [132]:
# list 타입이 들어있는 컬럼 찾기
list_cols = [
    c for c in panel_full_tab.columns
    if panel_full_tab[c].apply(lambda x: isinstance(x, list)).any()
]

print("list 타입 컬럼:", list_cols)

list 타입 컬럼: ['highway']


In [133]:
def list_to_first(x):
    if isinstance(x, list):
        return x[0] if len(x) > 0 else None
    return x

for c in list_cols:
    panel_full_tab[c] = panel_full_tab[c].apply(list_to_first)

print("list 컬럼 처리 완료")

path_panel_parquet = f"{out_dir}/panel_block_month_weather_proxy_label_v1_{ts}.parquet"

panel_full_tab.to_parquet(
    path_panel_parquet,
    index=False
)

print("Parquet 저장 완료:", path_panel_parquet)
print("Shape:", panel_full_tab.shape)

list 컬럼 처리 완료
Parquet 저장 완료: ./전처리데이터/panel_block_month_weather_proxy_label_v1_20251217_1433.parquet
Shape: (106320, 58)


# 모델 학습

## 데이터 로드 및 확인

In [4]:
import pandas as pd

path = "./전처리데이터/panel_block_month_weather_proxy_label_v1_20251217_1433.csv"

panel_full_tab = pd.read_csv(
    path,
    encoding="utf-8-sig"
)

print("Loaded shape:", panel_full_tab.shape)
panel_full_tab.head()

Loaded shape: (106320, 60)


,block_id,as_of_month,label_month,rain_sum_30d,rain_days_30d,freeze_days_30d,temp_range_30d,rain_sum_90d,rain_days_90d,freeze_days_90d,...,RN_CD,ROA_CLS_SE,SIG_CD,WDR_RD_CD,road_id,centroid,highway,OSM_HIGHWAY,ROAD_LEVEL,geom_buffer_30m
0,1000013_000_1000013_000_1,2024-01-01,2024-02-01,124.1,10.0,14.0,6.786667,176.6,21.0,28.0,...,1000013,1,30200,1,0,POINT (226806.71334983184 408317.6854960977),NaN,NaN,미분류,POLYGON ((226144.64875697208 407556.4384502459...
1,1000013_000_1000013_000_1,2024-02-01,2024-03-01,47.8,10.0,20.0,8.146923,212.2,29.0,49.0,...,1000013,1,30200,1,0,POINT (226806.71334983184 408317.6854960977),NaN,NaN,미분류,POLYGON ((226144.64875697208 407556.4384502459...
2,1000013_000_1000013_000_1,2024-03-01,2024-04-01,93.9,13.0,13.0,8.950000,265.8,33.0,48.0,...,1000013,1,30200,1,0,POINT (226806.71334983184 408317.6854960977),NaN,NaN,미분류,POLYGON ((226144.64875697208 407556.4384502459...
3,1000013_000_1000013_000_1,2024-04-01,2024-05-01,57.9,9.0,7.0,11.460000,199.6,32.0,41.0,...,1000013,1,30200,1,0,POINT (226806.71334983184 408317.6854960977),NaN,NaN,미분류,POLYGON ((226144.64875697208 407556.4384502459...
4,1000013_000_1000013_000_1,2024-05-01,2024-06-01,32.7,7.0,0.0,12.260000,184.5,29.0,21.0,...,1000013,1,30200,1,0,POINT (226806.71334983184 408317.6854960977),NaN,NaN,미분류,POLYGON ((226144.64875697208 407556.4384502459...


In [5]:
panel_full_tab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106320 entries, 0 to 106319
Data columns (total 60 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   block_id              106320 non-null  object 
 1   as_of_month           106320 non-null  object 
 2   label_month           106320 non-null  object 
 3   rain_sum_30d          106320 non-null  float64
 4   rain_days_30d         106320 non-null  float64
 5   freeze_days_30d       106320 non-null  float64
 6   temp_range_30d        106320 non-null  float64
 7   rain_sum_90d          106320 non-null  float64
 8   rain_days_90d         106320 non-null  float64
 9   freeze_days_90d       106320 non-null  float64
 10  next_event_cnt        106320 non-null  int64  
 11  next_risk_grade       106320 non-null  int64  
 12  length_m              106320 non-null  float64
 13  RDS_MAN_NO            106320 non-null  float64
 14  ROAD_BT               106320 non-null  float64
 15  

In [6]:
for c in ["as_of_month", "label_month"]:
    if c in panel_full_tab.columns:
        panel_full_tab[c] = pd.to_datetime(panel_full_tab[c])

print(panel_full_tab[["as_of_month", "label_month"]].dtypes)

as_of_month    datetime64[ns]
label_month    datetime64[ns]
dtype: object


In [7]:
print(panel_full_tab["next_risk_grade"].value_counts().sort_index())

next_risk_grade
0    105756
1       505
2        59
Name: count, dtype: int64


## XGBoost 학습

### 피쳐 구성

In [12]:
EXCLUDE_COLS = [
    # 타깃 / 누수
    "next_risk_grade",
    "next_event_cnt",

    # 식별자
    "block_id",
    "road_id",
    "section_id",
    "sub_id",

    # 시간 키 (index 용)
    "as_of_month",
    "label_month",

    # geometry / 공간 객체
    "centroid",
    "geom_buffer_30m",
]

weather_cols = [
    "rain_sum_30d",
    "rain_days_30d",
    "freeze_days_30d",
    "temp_range_30d",
    "rain_sum_90d",
    "rain_days_90d",
    "freeze_days_90d",
]

traffic_proxy_cols = [
    "traffic_load_proxy",
    "cap_combo",
    "heavy_vehicle_proxy",
    "intersection_cnt_30m",
]

road_numeric_cols = [
    "ROAD_WIDTH",
    "length_m",
    "road_length_m",
    "ROAD_BT",
    "ROAD_LT",
    "section_order",
    "dist_osm_m",
    "buffer_area",
]

categorical_try_cols = [
    "ROAD_LEVEL",      # 소/중/대로
    "OSM_HIGHWAY",     # primary / secondary / residential 등
    "ROA_CLS_SE",      # 도로 분류
]

In [13]:
feature_cols = (
    weather_cols
    + traffic_proxy_cols
    + road_numeric_cols
    + categorical_try_cols
)

### X, y 구성

In [14]:
df = panel_full_tab.copy()

y = df["next_risk_grade"].astype(int)

X = df[feature_cols].copy()

print("Before encoding:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True).sort_index())

Before encoding: (106320, 22)
y distribution:
 next_risk_grade
0    0.994695
1    0.004750
2    0.000555
Name: proportion, dtype: float64


In [ ]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Categorical columns:", cat_cols)

X = pd.get_dummies(
    X,
    columns=cat_cols,
    dummy_na=True
)

print("After encoding:", X.shape)

Categorical columns: ['ROAD_LEVEL', 'OSM_HIGHWAY']
After encoding: (106320, 38)


In [16]:
print("Final X shape:", X.shape)

Final X shape: (106320, 38)


### 학습

In [17]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# 시간 기준 정렬
df_sorted = panel_full_tab.sort_values("label_month").reset_index(drop=True)

cut = int(len(df_sorted) * 0.8)

X_all = X
y_all = y

X_train = X_all.iloc[:cut]
y_train = y_all.iloc[:cut]

X_test  = X_all.iloc[cut:]
y_test  = y_all.iloc[cut:]

print("Train period:", df_sorted["label_month"].iloc[0], "→", df_sorted["label_month"].iloc[cut-1])
print("Test  period:", df_sorted["label_month"].iloc[cut], "→", df_sorted["label_month"].iloc[-1])

print("Train shape:", X_train.shape)
print("Test  shape:", X_test.shape)

Train period: 2024-02-01 00:00:00 → 2025-09-01 00:00:00
Test  period: 2025-09-01 00:00:00 → 2026-01-01 00:00:00
Train shape: (85056, 38)
Test  shape: (21264, 38)


- 클래스 불균형 처리를 위해 가중치 적용

In [18]:
class_counts = y_train.value_counts().sort_index()
print("Train class counts:\n", class_counts)

# 기준: class 0 대비 가중치
scale_pos_weight_1 = class_counts[0] / class_counts[1]
scale_pos_weight_2 = class_counts[0] / class_counts[2]

print(f"scale_pos_weight class 1: {scale_pos_weight_1:.1f}")
print(f"scale_pos_weight class 2: {scale_pos_weight_2:.1f}")

Train class counts:
 next_risk_grade
0    84532
1      466
2       58
Name: count, dtype: int64
scale_pos_weight class 1: 181.4
scale_pos_weight class 2: 1457.4


In [19]:
# sample_weight 생성
sample_weight = y_train.map({
    0: 1.0,
    1: scale_pos_weight_1,
    2: scale_pos_weight_2
}).values

In [22]:
dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    weight=sample_weight
)

dtest = xgb.DMatrix(
    X_test,
    label=y_test
)

params = {
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",

    # 트리 설정
    "tree_method": "hist",   # 🔥 gpu_hist 제거
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,

    # 불균형 대응
    "max_delta_step": 1,

    # CPU 병렬
    "nthread": -1,

    # 재현성
    "seed": 42
}

In [23]:
evals = [(dtrain, "train"), (dtest, "test")]

num_boost_round = 500
early_stopping_rounds = 50

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=evals,
    early_stopping_rounds=early_stopping_rounds,
    verbose_eval=10  # 🔥 10라운드마다 loss 출력
)

[0]	train-mlogloss:1.06008	test-mlogloss:1.05472
[10]	train-mlogloss:0.77778	test-mlogloss:0.69484
[20]	train-mlogloss:0.59835	test-mlogloss:0.48419
[30]	train-mlogloss:0.48840	test-mlogloss:0.36364
[40]	train-mlogloss:0.41275	test-mlogloss:0.28936
[50]	train-mlogloss:0.35955	test-mlogloss:0.24442
[60]	train-mlogloss:0.31712	test-mlogloss:0.21285
[70]	train-mlogloss:0.28425	test-mlogloss:0.19103
[80]	train-mlogloss:0.25662	test-mlogloss:0.17604
[90]	train-mlogloss:0.23336	test-mlogloss:0.16520
[100]	train-mlogloss:0.21181	test-mlogloss:0.15513
[110]	train-mlogloss:0.19395	test-mlogloss:0.14893
[120]	train-mlogloss:0.17945	test-mlogloss:0.14111
[130]	train-mlogloss:0.16605	test-mlogloss:0.13471
[140]	train-mlogloss:0.15397	test-mlogloss:0.12918
[150]	train-mlogloss:0.14249	test-mlogloss:0.12329
[160]	train-mlogloss:0.13327	test-mlogloss:0.11945
[170]	train-mlogloss:0.12557	test-mlogloss:0.11409
[180]	train-mlogloss:0.11726	test-mlogloss:0.10857
[190]	train-mlogloss:0.10989	test-mlogloss

In [24]:
# 예측
proba = model.predict(dtest)
pred = proba.argmax(axis=1)

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, pred))

print("\n=== Classification Report ===")
print(classification_report(y_test, pred, digits=4))


=== Confusion Matrix ===
[[21012   211     1]
 [   37     2     0]
 [    0     1     0]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9982    0.9900    0.9941     21224
           1     0.0093    0.0513    0.0158        39
           2     0.0000    0.0000    0.0000         1

    accuracy                         0.9882     21264
   macro avg     0.3359    0.3471    0.3366     21264
weighted avg     0.9964    0.9882    0.9923     21264



In [25]:
for cls in [1, 2]:
    y_bin = (y_test.values == cls).astype(int)
    ap = average_precision_score(y_bin, proba[:, cls])
    print(f"PR-AUC (class {cls}): {ap:.6f}")

PR-AUC (class 1): 0.007249
PR-AUC (class 2): 0.005587


### threshold 튜닝 후 재평가

In [26]:
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, classification_report

# class 1을 '양성'으로 보고 threshold 탐색
cls = 1
y_bin = (y_test.values == cls).astype(int)
p = proba[:, cls]

prec, rec, thr = precision_recall_curve(y_bin, p)
ap = average_precision_score(y_bin, p)

print(f"AP(PR-AUC) for class {cls}: {ap:.6f}")
print("threshold candidates:", len(thr))

# F1 최대가 되는 threshold 찾기
f1 = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_i = int(np.argmax(f1))
best_thr = thr[best_i]
print(f"Best threshold for class {cls} by F1: {best_thr:.6f}")
print(f"  Precision={prec[best_i]:.4f}, Recall={rec[best_i]:.4f}, F1={f1[best_i]:.4f}")

# threshold 적용한 예측: p>=thr면 class1, 아니면 class0
pred_thr = np.where(p >= best_thr, 1, 0)

print("\n=== Confusion matrix (0 vs 1, thresholded) ===")
print(confusion_matrix(y_bin, (pred_thr==1).astype(int)))

print("\n=== Classification report (0 vs 1, thresholded) ===")
print(classification_report(y_bin, (pred_thr==1).astype(int), digits=4))

AP(PR-AUC) for class 1: 0.007249
threshold candidates: 21261
Best threshold for class 1 by F1: 0.430223
  Precision=0.0148, Recall=0.1026, F1=0.0259

=== Confusion matrix (0 vs 1, thresholded) ===
[[20959   266]
 [   35     4]]

=== Classification report (0 vs 1, thresholded) ===
              precision    recall  f1-score   support

           0     0.9983    0.9875    0.9929     21225
           1     0.0148    0.1026    0.0259        39

    accuracy                         0.9858     21264
   macro avg     0.5066    0.5450    0.5094     21264
weighted avg     0.9965    0.9858    0.9911     21264



### 라벨 재정의 후 재학습
- 기존 라벨 : 0(좋음), 2(보통), 3(나쁨)
- 새 라벨: 0(정상), 1(위험)

In [27]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, precision_recall_curve

# 1) 이진 타깃: 위험(1/2)=1, 정상(0)=0
y_train_bin = (y_train.values >= 1).astype(int)
y_test_bin  = (y_test.values >= 1).astype(int)

print("Train positives:", y_train_bin.sum(), "/", len(y_train_bin))
print("Test  positives:", y_test_bin.sum(), "/", len(y_test_bin))

# 2) DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train_bin)
dtest  = xgb.DMatrix(X_test,  label=y_test_bin)

# 3) 불균형 가중치 (음성/양성 비율)
neg = (y_train_bin == 0).sum()
pos = (y_train_bin == 1).sum()
scale_pos_weight = neg / max(pos, 1)
print("scale_pos_weight:", scale_pos_weight)

# 4) 파라미터 (AUC-PR 중심)
params = {
    "objective": "binary:logistic",
    "eval_metric": ["aucpr", "logloss"],
    "eta": 0.05,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "alpha": 0.0,
    "scale_pos_weight": scale_pos_weight,
    "tree_method": "hist",   # CPU 안정
    "seed": 42,
}

evals = [(dtrain, "train"), (dtest, "test")]
evals_result = {}

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=evals,
    early_stopping_rounds=100,
    verbose_eval=25,     # 25 라운드마다 로그(진행상황/로스)
    evals_result=evals_result
)

print("best_iteration:", model.best_iteration)
print("best_score:", model.best_score)

# 5) 예측 확률 + 기본 임계값(0.5) 결과
p = model.predict(dtest)
pred05 = (p >= 0.5).astype(int)

print("\n=== Confusion (thr=0.5) ===")
print(confusion_matrix(y_test_bin, pred05))
print(classification_report(y_test_bin, pred05, digits=4))
print("AP(AUC-PR):", average_precision_score(y_test_bin, p))

# 6) F1 기준 최적 threshold도 같이 출력 (원하면 운영 임계값으로 사용)
prec, rec, thr = precision_recall_curve(y_test_bin, p)
f1 = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_i = int(np.argmax(f1))
best_thr = thr[best_i]
print("\nBest threshold by F1:", float(best_thr))
print(f"Precision={prec[best_i]:.4f}, Recall={rec[best_i]:.4f}, F1={f1[best_i]:.4f}")

pred_best = (p >= best_thr).astype(int)
print("\n=== Confusion (best_thr) ===")
print(confusion_matrix(y_test_bin, pred_best))
print(classification_report(y_test_bin, pred_best, digits=4))

Train positives: 524 / 85056
Test  positives: 40 / 21264
scale_pos_weight: 161.3206106870229
[0]	train-aucpr:0.09704	train-logloss:0.66100	test-aucpr:0.01203	test-logloss:0.65620
[25]	train-aucpr:0.21436	train-logloss:0.31782	test-aucpr:0.01416	test-logloss:0.27876
[50]	train-aucpr:0.25168	train-logloss:0.22947	test-aucpr:0.01453	test-logloss:0.17823
[75]	train-aucpr:0.29038	train-logloss:0.19492	test-aucpr:0.01362	test-logloss:0.14604
[100]	train-aucpr:0.31793	train-logloss:0.17401	test-aucpr:0.01280	test-logloss:0.12720
[125]	train-aucpr:0.34232	train-logloss:0.15591	test-aucpr:0.01213	test-logloss:0.11227
[150]	train-aucpr:0.36813	train-logloss:0.14484	test-aucpr:0.01176	test-logloss:0.10280
[175]	train-aucpr:0.39987	train-logloss:0.13086	test-aucpr:0.01171	test-logloss:0.09058
[200]	train-aucpr:0.44334	train-logloss:0.11790	test-aucpr:0.01178	test-logloss:0.07955
[225]	train-aucpr:0.48415	train-logloss:0.10719	test-aucpr:0.01187	test-logloss:0.07115
[250]	train-aucpr:0.51570	train-

### 데이터 불균형으로인한 학습 성능 저하 원인 분석

In [28]:
import numpy as np
import pandas as pd

df = panel_full_tab.copy()

# 0) 기본 타입 정리
for c in ["as_of_month", "label_month"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

y = df["next_risk_grade"].astype(int)

print("=== [0] 전체 크기/라벨 분포 ===")
print("shape:", df.shape)
print("label_month range:", df["label_month"].min(), "→", df["label_month"].max())
print("y counts:\n", y.value_counts().sort_index())
print("y ratio:\n", y.value_counts(normalize=True).sort_index())

# 1) 월별 양성 비율(시간 신호 존재 여부)
print("\n=== [1] 월별 양성(>=1) 발생률 추이 ===")
df["y_pos"] = (df["next_risk_grade"] >= 1).astype(int)
monthly = df.groupby("label_month").agg(
    n=("y_pos", "size"),
    pos=("y_pos", "sum"),
)
monthly["pos_rate"] = monthly["pos"] / monthly["n"]
print(monthly.tail(18))  # 최근 18개월 정도만

print("\n월별 pos_rate 요약:")
print(monthly["pos_rate"].describe())

# 2) 블록별 사건 집중도(특정 블록에만 반복되면 '블록 고정 신호'가 큼)
print("\n=== [2] 블록별 양성(>=1) 월 개수 분포 ===")
blk = df.groupby("block_id")["y_pos"].sum().sort_values(ascending=False)
print("block_id unique:", df["block_id"].nunique())
print("양성 월이 1회 이상인 블록 수:", (blk >= 1).sum())
print("양성 월이 2회 이상인 블록 수:", (blk >= 2).sum())
print("상위 20 블록(양성 월 개수):")
print(blk.head(20))

print("\n블록별 양성 월 개수 요약:")
print(blk.describe())

# 3) 피처가 '고정값'인지 확인 (월별 변동 없는 컬럼이 많으면 예측력 제한)
#    - 수치형 컬럼 중에서 label_month 그룹 내 분산이 0에 가까운 컬럼을 찾음
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
drop_num = {"next_risk_grade", "next_event_cnt", "y_pos"}  # 타깃/파생 제거
num_cols = [c for c in num_cols if c not in drop_num]

print("\n=== [3] (수치) 월별로 거의 변하지 않는 컬럼 TOP ===")
# 월별 표준편차 평균이 매우 작은 컬럼
std_by_month = df.groupby("label_month")[num_cols].std(numeric_only=True)
mean_std = std_by_month.mean().sort_values()
print("mean(std within month) 최하위 15개:")
print(mean_std.head(15))

print("\n=== [4] (수치) 블록별로 거의 변하지 않는 컬럼 TOP ===")
std_by_block = df.groupby("block_id")[num_cols].std(numeric_only=True)
mean_std_blk = std_by_block.mean().sort_values()
print("mean(std within block) 최하위 15개:")
print(mean_std_blk.head(15))

# 4) 간단한 '단변량 신호' 체크: 상위 proxy/기상 피처와 양성의 평균 비교
cand = [c for c in [
    "traffic_load_proxy", "cap_combo", "heavy_vehicle_proxy", "intersection_cnt_30m",
    "rain_sum_30d", "rain_days_30d", "freeze_days_30d", "temp_range_30d",
    "rain_sum_90d", "rain_days_90d", "freeze_days_90d"
] if c in df.columns]

print("\n=== [5] 주요 피처 단변량 평균 비교 (y_pos=0 vs 1) ===")
if cand:
    comp = df.groupby("y_pos")[cand].mean()
    print(comp)
    print("\n(차이: pos - neg)")
    print((comp.loc[1] - comp.loc[0]).sort_values(ascending=False))
else:
    print("cand 피처가 없음")

# 정리
print("\n[완료] 위 출력 결과를 그대로 공유해 주세요. 다음 단계(우선순위 2)로 바로 이어갑니다.")

=== [0] 전체 크기/라벨 분포 ===
shape: (106320, 60)
label_month range: 2024-02-01 00:00:00 → 2026-01-01 00:00:00
y counts:
 next_risk_grade
0    105756
1       505
2        59
Name: count, dtype: int64
y ratio:
 next_risk_grade
0    0.994695
1    0.004750
2    0.000555
Name: proportion, dtype: float64

=== [1] 월별 양성(>=1) 발생률 추이 ===
                n  pos  pos_rate
label_month                     
2024-08-01   4430   13  0.002935
2024-09-01   4430    9  0.002032
2024-10-01   4430   15  0.003386
2024-11-01   4430    7  0.001580
2024-12-01   4430   24  0.005418
2025-01-01   4430   22  0.004966
2025-02-01   4430   14  0.003160
2025-03-01   4430    0  0.000000
2025-04-01   4430    0  0.000000
2025-05-01   4430    0  0.000000
2025-06-01   4430    0  0.000000
2025-07-01   4430    0  0.000000
2025-08-01   4430    0  0.000000
2025-09-01   4430    0  0.000000
2025-10-01   4430    6  0.001354
2025-11-01   4430   47  0.010609
2025-12-01   4430   44  0.009932
2026-01-01   4430    0  0.000000

월별 pos_rate 요

### 언더샘플링 XGBoost 실행

In [30]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)
from tqdm import tqdm

df = panel_full_tab.copy()

y = (df["next_risk_grade"] >= 1).astype(int)

print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

y distribution:
next_risk_grade
0    105756
1       564
Name: count, dtype: int64
next_risk_grade
0    0.994695
1    0.005305
Name: proportion, dtype: float64


In [35]:
# 1) 기본 DF
df = panel_full_tab.copy()

# 2) 타깃 (binary: 0 vs >=1)
y = (df["next_risk_grade"] >= 1).astype(int)

# 3) 피처만 선택
X = df[feature_cols].copy()

print("Before encoding X shape:", X.shape)

# 4) categorical 처리
cat_cols = [c for c in categorical_try_cols if c in X.columns]
print("Categorical cols:", cat_cols)

X = pd.get_dummies(X, columns=cat_cols, dummy_na=True)

print("After encoding X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))

Before encoding X shape: (106320, 22)
Categorical cols: ['ROAD_LEVEL', 'OSM_HIGHWAY', 'ROA_CLS_SE']
After encoding X shape: (106320, 42)
y distribution:
 next_risk_grade
0    0.994695
1    0.005305
Name: proportion, dtype: float64


In [36]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, precision_recall_curve
)

# ----------------------------
# 0) 데이터 준비
# ----------------------------
df = panel_full_tab.copy()

# 타깃: 다음달에 1회 이상 발생이면 1
y_all = (df["next_risk_grade"] >= 1).astype(int)

# 피처: 이미 네가 만든 22개(원-핫 전)
feature_cols = [
    "rain_sum_30d","rain_days_30d","freeze_days_30d","temp_range_30d",
    "rain_sum_90d","rain_days_90d","freeze_days_90d",
    "traffic_load_proxy","cap_combo","heavy_vehicle_proxy","intersection_cnt_30m",
    "ROAD_WIDTH","length_m","road_length_m","ROAD_BT","ROAD_LT","section_order","dist_osm_m","buffer_area",
    "ROAD_LEVEL","OSM_HIGHWAY","ROA_CLS_SE",
]
X_raw = df[feature_cols].copy()

# 범주형 원-핫
cat_cols = ["ROAD_LEVEL","OSM_HIGHWAY","ROA_CLS_SE"]
X_all = pd.get_dummies(X_raw, columns=cat_cols, dummy_na=True)

# feature name 안전화(대괄호/꺽쇠 문제 방지)
X_all.columns = (
    X_all.columns.astype(str)
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

# ----------------------------
# 1) 시간 기준 split (label_month 기준)
# ----------------------------
df_sorted = df.copy()
df_sorted["y"] = y_all.values
df_sorted = df_sorted.sort_values("label_month").reset_index(drop=True)

X_sorted = X_all.loc[df_sorted.index].reset_index(drop=True)
y_sorted = df_sorted["y"].astype(int).reset_index(drop=True)

cut = int(len(df_sorted) * 0.8)

X_train, y_train = X_sorted.iloc[:cut], y_sorted.iloc[:cut]
X_test,  y_test  = X_sorted.iloc[cut:], y_sorted.iloc[cut:]

print("Train months:", df_sorted["label_month"].iloc[0], "→", df_sorted["label_month"].iloc[cut-1])
print("Test  months:", df_sorted["label_month"].iloc[cut], "→", df_sorted["label_month"].iloc[-1])
print("Train pos rate:", y_train.mean(), "Test pos rate:", y_test.mean())

# ----------------------------
# 2) 언더샘플링 (train에서만!)
#    pos:neg = 1:10 추천 (너무 줄이면 학습 불안정)
# ----------------------------
pos_idx = np.where(y_train.values == 1)[0]
neg_idx = np.where(y_train.values == 0)[0]

np.random.seed(42)

neg_keep = min(len(neg_idx), len(pos_idx) * 10)  # 1:10
neg_keep_idx = np.random.choice(neg_idx, size=neg_keep, replace=False)

keep_idx = np.concatenate([pos_idx, neg_keep_idx])
np.random.shuffle(keep_idx)

Xtr = X_train.iloc[keep_idx]
ytr = y_train.iloc[keep_idx]

print("\n[Under-sampling train only]")
print("Before:", len(y_train), "pos:", int(y_train.sum()), "neg:", int((y_train==0).sum()))
print("After :", len(ytr),     "pos:", int(ytr.sum()),     "neg:", int((ytr==0).sum()))

# ----------------------------
# 3) DMatrix
# ----------------------------
dtrain = xgb.DMatrix(Xtr, label=ytr)
dtest  = xgb.DMatrix(X_test, label=y_test)

# ----------------------------
# 4) XGBoost 파라미터
#    - 너 환경에서 gpu_hist가 안 먹었으니 hist로 고정
# ----------------------------
params = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "aucpr"],   # aucpr이 핵심
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "reg_lambda": 1.0,
    "tree_method": "hist",   # GPU 안 되면 hist
    "seed": 42,
}

evals = [(dtrain, "train"), (dtest, "test")]

num_boost_round = 3000
early_stopping_rounds = 100

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    evals=evals,
    early_stopping_rounds=early_stopping_rounds,
    verbose_eval=25  # 25 라운드마다 진행상황 출력
)

print("\nbest_iteration:", model.best_iteration)
print("best_score:", model.best_score)

# ----------------------------
# 5) 평가: PR-AUC + 임계값 최적화
# ----------------------------
proba = model.predict(dtest)
ap = average_precision_score(y_test.values, proba)
print("\nAP(AUC-PR):", ap)

# 기본 0.5
pred_05 = (proba >= 0.5).astype(int)
print("\n=== Confusion (thr=0.5) ===")
print(confusion_matrix(y_test, pred_05))
print(classification_report(y_test, pred_05, digits=4))

# F1 최대 threshold 탐색
prec, rec, thr = precision_recall_curve(y_test.values, proba)
f1 = (2*prec*rec) / (prec + rec + 1e-12)
best_i = np.argmax(f1)
best_thr = thr[best_i] if best_i < len(thr) else 0.5

print("\nBest threshold by F1:", best_thr)
print(f"Precision={prec[best_i]:.4f}, Recall={rec[best_i]:.4f}, F1={f1[best_i]:.4f}")

pred_best = (proba >= best_thr).astype(int)
print("\n=== Confusion (best_thr) ===")
print(confusion_matrix(y_test, pred_best))
print(classification_report(y_test, pred_best, digits=4))

Train months: 2024-02-01 00:00:00 → 2025-09-01 00:00:00
Test  months: 2025-09-01 00:00:00 → 2026-01-01 00:00:00
Train pos rate: 0.005490500376222724 Test pos rate: 0.004561700526711813

[Under-sampling train only]
Before: 85056 pos: 467 neg: 84589
After : 5137 pos: 467 neg: 4670
[0]	train-logloss:0.29886	train-aucpr:0.28987	test-logloss:0.10305	test-aucpr:0.00486
[25]	train-logloss:0.24271	train-aucpr:0.44501	test-logloss:0.06480	test-aucpr:0.00565
[50]	train-logloss:0.22254	train-aucpr:0.51338	test-logloss:0.05307	test-aucpr:0.00584
[75]	train-logloss:0.21016	train-aucpr:0.56453	test-logloss:0.05061	test-aucpr:0.00575
[100]	train-logloss:0.19963	train-aucpr:0.61011	test-logloss:0.04896	test-aucpr:0.00576
[125]	train-logloss:0.19047	train-aucpr:0.65364	test-logloss:0.04872	test-aucpr:0.00590
[150]	train-logloss:0.18093	train-aucpr:0.69426	test-logloss:0.04823	test-aucpr:0.00570
[175]	train-logloss:0.17160	train-aucpr:0.73271	test-logloss:0.04814	test-aucpr:0.00559
[200]	train-logloss:0

c:\Users\hyebi\anaconda3\envs\EDA\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hyebi\anaconda3\envs\EDA\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hyebi\anaconda3\envs\EDA\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# 데이터 재구성 (라벨 및 피처 설계 보강)

## 기존 통합 데이터 로드

In [44]:
import pandas as pd

path = "./전처리데이터/panel_block_month_weather_proxy_label_v1_20251217_1433.csv"

panel = pd.read_csv(path, encoding="utf-8-sig")

# 날짜 컬럼 정리 (안 돼 있으면 반드시)
panel["as_of_month"] = pd.to_datetime(panel["as_of_month"])
panel["label_month"] = pd.to_datetime(panel["label_month"])

print("Loaded shape:", panel.shape)
print("Columns:", panel.columns.tolist())
print("Month range:", panel["as_of_month"].min(), "→", panel["as_of_month"].max())
print("Label dist:\n", panel["next_risk_grade"].value_counts())

Loaded shape: (106320, 60)
Columns: ['block_id', 'as_of_month', 'label_month', 'rain_sum_30d', 'rain_days_30d', 'freeze_days_30d', 'temp_range_30d', 'rain_sum_90d', 'rain_days_90d', 'freeze_days_90d', 'next_event_cnt', 'next_risk_grade', 'length_m', 'RDS_MAN_NO', 'ROAD_BT', 'ROAD_LT', 'road_length_m', 'section_order', 'u', 'v', 'key', 'dist_osm_m', 'ROAD_WIDTH', 'cap_level_score', 'cap_width', 'cap_combo', 'heavy_osm_score', 'heavy_level_score', 'heavy_vehicle_proxy', 'buffer_area', 'intersection_cnt_30m', 'cap_norm', 'heavy_norm', 'intersect_norm', 'traffic_load_proxy', 'section_id', 'sub_id', 'ALWNC_DE', 'ALWNC_RESN', 'BSI_INT', 'ENG_RN', 'MVMN_DE', 'MVMN_RESN', 'MVM_RES_CD', 'NTFC_DE', 'OPERT_DE', 'RBP_CN', 'RDS_DPN_SE', 'REP_CN', 'RN', 'RN_CD', 'ROA_CLS_SE', 'SIG_CD', 'WDR_RD_CD', 'road_id', 'centroid', 'highway', 'OSM_HIGHWAY', 'ROAD_LEVEL', 'geom_buffer_30m']
Month range: 2024-01-01 00:00:00 → 2025-12-01 00:00:00
Label dist:
 next_risk_grade
0    105756
1       505
2        59
Na

## 1) 블록 단위 유지 + 라벨 윈도우 확장

### 라벨 생성 및 데이터 정제

In [82]:
import pandas as pd
import numpy as np

panel = panel.sort_values(["block_id", "as_of_month"]).reset_index(drop=True)

g = panel.groupby("block_id")["next_event_cnt"]

# next 3개월 / 6개월 이벤트 합
panel["next_3m_event_sum"] = (
    g.shift(-1).fillna(0)
  + g.shift(-2).fillna(0)
  + g.shift(-3).fillna(0)
)

panel["next_6m_event_sum"] = (
    g.shift(-1).fillna(0)
  + g.shift(-2).fillna(0)
  + g.shift(-3).fillna(0)
  + g.shift(-4).fillna(0)
  + g.shift(-5).fillna(0)
  + g.shift(-6).fillna(0)
)

# 이진 라벨 정의
panel["y_3m"] = (panel["next_3m_event_sum"] >= 1).astype(int)
panel["y_6m"] = (panel["next_6m_event_sum"] >= 1).astype(int)

print("y_3m dist:\n", panel["y_3m"].value_counts())
print("y_6m dist:\n", panel["y_6m"].value_counts())

y_3m dist:
 y_3m
0    105332
1       988
Name: count, dtype: int64
y_6m dist:
 y_6m
0    104836
1      1484
Name: count, dtype: int64


In [46]:
# 라벨/미래정보/ID 제거
DROP_COLS = [
    "block_id", "as_of_month", "label_month",
    "next_event_cnt", "next_risk_grade",
    "next_3m_event_sum", "next_6m_event_sum",
    "y_3m", "y_6m"
]

X_cols = [c for c in panel.columns if c not in DROP_COLS]
print("Num features:", len(X_cols))

Num features: 55


In [51]:
# 기준 월 설정 (예시: 마지막 6개월 테스트)
max_month = panel["as_of_month"].max()
test_start = max_month - pd.DateOffset(months=6)

train_df = panel[panel["as_of_month"] < test_start]
test_df  = panel[panel["as_of_month"] >= test_start]

X_train = train_df[X_cols]
y_train = train_df["y_3m"]    # ⭐ next 3개월 기준
X_test  = test_df[X_cols]
y_test  = test_df["y_3m"]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train y dist:\n", y_train.value_counts())
print("Test y dist:\n", y_test.value_counts())

Train: (75310, 55) Test: (31010, 55)
Train y dist:
 y_3m
0    74564
1      746
Name: count, dtype: int64
Test y dist:
 y_3m
0    30768
1      242
Name: count, dtype: int64


In [53]:
leak_cols = [c for c in X_cols if c.startswith("next_") or c in ["next_event_cnt", "next_risk_grade", "y_3m", "y_6m"]]
print("Leak-like cols in X:", leak_cols)

Leak-like cols in X: []


In [54]:
na_rate = X_train.isna().mean().sort_values(ascending=False).head(10)
print(na_rate)

dist_osm_m       0.240181
v                0.240181
u                0.240181
key              0.240181
OSM_HIGHWAY      0.240181
highway          0.240181
MVMN_RESN        0.039052
MVM_RES_CD       0.004515
rain_days_90d    0.000000
rain_sum_90d     0.000000
dtype: float64


In [55]:
# 1) 제거 대상(식별자/그래프 키)
drop_cols = [c for c in ["u", "v", "key"] if c in panel.columns]
panel = panel.drop(columns=drop_cols, errors="ignore")

# 2) OSM 거리: 결측은 "매칭 실패"로 해석 가능 -> 큰 값으로 채우거나 아예 제거
if "dist_osm_m" in panel.columns:
    # 선택 A(권장): 큰 값으로 치환해 '멀다/없다' 신호를 남김
    panel["dist_osm_m"] = panel["dist_osm_m"].fillna(9999)

    # 선택 B: 그냥 제거하고 싶으면 아래 줄 사용
    # panel = panel.drop(columns=["dist_osm_m"])

# 3) 카테고리 결측은 unknown으로
for c in ["OSM_HIGHWAY", "highway", "MVMN_RESN", "MVM_RES_CD"]:
    if c in panel.columns:
        panel[c] = panel[c].astype("object").fillna("unknown")

# (권장) OSM 매칭 성공 여부 플래그(신호가 될 수 있음)
if "dist_osm_m" in panel.columns:
    panel["has_osm_match"] = (panel["dist_osm_m"] < 9999).astype(int)

In [56]:
DROP_COLS = [
    "block_id", "as_of_month", "label_month",
    "next_event_cnt", "next_risk_grade",
    "next_3m_event_sum", "next_6m_event_sum",
    "y_3m", "y_6m"
]
X_cols = [c for c in panel.columns if c not in DROP_COLS]

# 분할(당신이 쓰던 기준 그대로)
max_month = panel["as_of_month"].max()
test_start = max_month - pd.DateOffset(months=6)

train_df = panel[panel["as_of_month"] < test_start]
test_df  = panel[panel["as_of_month"] >= test_start]

X_train = train_df[X_cols]
y_train = train_df["y_3m"]
X_test  = test_df[X_cols]
y_test  = test_df["y_3m"]

print("Top NA rates:\n", X_train.isna().mean().sort_values(ascending=False).head(10))

Top NA rates:
 rain_sum_30d       0.0
rain_days_30d      0.0
freeze_days_30d    0.0
temp_range_30d     0.0
rain_sum_90d       0.0
rain_days_90d      0.0
freeze_days_90d    0.0
length_m           0.0
RDS_MAN_NO         0.0
ROAD_BT            0.0
dtype: float64


In [59]:
DROP_GEOM_COLS = ["centroid", "geom_buffer_30m"]

panel = panel.drop(columns=[c for c in DROP_GEOM_COLS if c in panel.columns])

In [60]:
def sanitize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace(r"[\[\]<]", "_", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

### y_3m으로 학습

In [65]:
# ================================
# Train 내부 Validation 분리 (최근 3개월)
# ================================
val_start = train_df["as_of_month"].max() - pd.DateOffset(months=3)

X_tr = X_train[train_df["as_of_month"] < val_start].copy()
y_tr = y_train[train_df["as_of_month"] < val_start].copy()

X_val = X_train[train_df["as_of_month"] >= val_start].copy()
y_val = y_train[train_df["as_of_month"] >= val_start].copy()

print("Train split:", X_tr.shape)
print("Valid split:", X_val.shape)

Train split: (57590, 53)
Valid split: (17720, 53)


In [66]:
# ================================
# 범주형 컬럼 처리
# ================================
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
print("Categorical cols:", cat_cols)

MIN_FREQ = 50

for c in cat_cols:
    vc = X_tr[c].value_counts()
    rare = vc[vc < MIN_FREQ].index

    X_tr[c]  = X_tr[c].where(~X_tr[c].isin(rare), other="other")
    X_val[c] = X_val[c].where(~X_val[c].isin(rare), other="other")
    X_test[c]= X_test[c].where(~X_test[c].isin(rare), other="other")

Categorical cols: ['section_id', 'sub_id', 'ALWNC_RESN', 'ENG_RN', 'MVMN_RESN', 'RBP_CN', 'REP_CN', 'RN', 'highway', 'OSM_HIGHWAY', 'ROAD_LEVEL']


In [67]:
# ================================
# 원-핫 인코딩
# ================================
X_tr_enc  = pd.get_dummies(X_tr,  columns=cat_cols, drop_first=False)
X_val_enc = pd.get_dummies(X_val, columns=cat_cols, drop_first=False)
X_test_enc= pd.get_dummies(X_test,columns=cat_cols, drop_first=False)

# train 기준으로 컬럼 정렬
X_val_enc  = X_val_enc.reindex(columns=X_tr_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_tr_enc.columns, fill_value=0)

In [68]:
def sanitize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace(r"[\[\]<]", "_", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

X_tr_enc   = sanitize_columns(X_tr_enc)
X_val_enc  = sanitize_columns(X_val_enc)
X_test_enc = sanitize_columns(X_test_enc)

print("Encoded shapes:",
      X_tr_enc.shape, X_val_enc.shape, X_test_enc.shape)

Encoded shapes: (57590, 2247) (17720, 2247) (31010, 2247)


In [69]:
pos = y_tr.sum()
neg = len(y_tr) - pos
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 76.19839142091153


In [73]:
from xgboost import XGBClassifier
from xgboost.callback import TrainingCallback
from tqdm.auto import tqdm

class TQDMCallback(TrainingCallback):
    def __init__(self, total_rounds):
        self.pbar = tqdm(total=total_rounds, desc="Training", leave=True)

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        if "validation_0" in evals_log:
            msg = []
            for m, v in evals_log["validation_0"].items():
                msg.append(f"{m}:{v[-1]:.5f}")
            self.pbar.set_postfix_str(" | ".join(msg))
        return False

    def after_training(self, model):
        self.pbar.close()


N_ESTIMATORS = 800

model = XGBClassifier(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    eval_metric=["aucpr", "logloss"],
    random_state=42,
)

model.fit(
    X_tr_enc, y_tr,
    verbose=50
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,"['aucpr', 'logloss']"


In [77]:
# object dtype 컬럼 확인
obj_cols_tr = X_tr_enc.select_dtypes(include=["object"]).columns.tolist()
obj_cols_val = X_val_enc.select_dtypes(include=["object"]).columns.tolist()
obj_cols_test = X_test_enc.select_dtypes(include=["object"]).columns.tolist()

print("Object cols (train):", obj_cols_tr)
print("Object cols (val):", obj_cols_val)
print("Object cols (test):", obj_cols_test)

Object cols (train): []
Object cols (val): []
Object cols (test): ['MVM_RES_CD']


In [78]:
def force_numeric(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype("category").cat.codes
    return df

X_tr_enc   = force_numeric(X_tr_enc)
X_val_enc  = force_numeric(X_val_enc)
X_test_enc = force_numeric(X_test_enc)

In [79]:
y_prob = model.predict_proba(X_test_enc)[:, 1]

print("y_prob range:", y_prob.min(), y_prob.max())

y_prob range: 1.0711138e-07 0.96275085


In [80]:
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np

print("PR-AUC:", average_precision_score(y_test, y_prob))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

def recall_at_k(y_true, y_score, k_ratio=0.05):
    k = max(1, int(len(y_score) * k_ratio))
    idx = np.argsort(y_score)[::-1][:k]
    return y_true.iloc[idx].sum() / max(1, y_true.sum())

for k in [0.01, 0.03, 0.05, 0.10]:
    print(f"Recall@Top{k*100:.0f}%:", recall_at_k(y_test, y_prob, k))

PR-AUC: 0.07595864166895794
ROC-AUC: 0.8363332570492903
Recall@Top1%: 0.1859504132231405
Recall@Top3%: 0.3347107438016529
Recall@Top5%: 0.43388429752066116
Recall@Top10%: 0.5826446280991735


### y_6m으로 학습

In [84]:
# block_id, as_of_month 정렬 (안 돼 있으면 필수)
panel = panel.sort_values(["block_id", "as_of_month"]).reset_index(drop=True)

g = panel.groupby("block_id")["next_event_cnt"]

# next 6개월 이벤트 합 (없으면 다시 생성)
panel["next_6m_event_sum"] = (
    g.shift(-1).fillna(0)
  + g.shift(-2).fillna(0)
  + g.shift(-3).fillna(0)
  + g.shift(-4).fillna(0)
  + g.shift(-5).fillna(0)
  + g.shift(-6).fillna(0)
)

# y_6m 이진 라벨
panel["y_6m"] = (panel["next_6m_event_sum"] >= 1).astype(int)

print("y_6m dist:\n", panel["y_6m"].value_counts())

y_6m dist:
 y_6m
0    104836
1      1484
Name: count, dtype: int64


In [85]:
# 다시 분할 (기존 기준 그대로)
max_month = panel["as_of_month"].max()
test_start = max_month - pd.DateOffset(months=6)

train_df = panel[panel["as_of_month"] < test_start].copy()
test_df  = panel[panel["as_of_month"] >= test_start].copy()

print("Train y_6m dist:\n", train_df["y_6m"].value_counts())
print("Test  y_6m dist:\n", test_df["y_6m"].value_counts())

Train y_6m dist:
 y_6m
0    74156
1     1154
Name: count, dtype: int64
Test  y_6m dist:
 y_6m
0    30680
1      330
Name: count, dtype: int64


In [86]:
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
MIN_FREQ = 50

for c in cat_cols:
    vc = X_tr[c].value_counts()
    rare = vc[vc < MIN_FREQ].index

    X_tr[c]  = X_tr[c].where(~X_tr[c].isin(rare), other="other")
    X_val[c] = X_val[c].where(~X_val[c].isin(rare), other="other")
    X_test[c]= X_test[c].where(~X_test[c].isin(rare), other="other")

X_tr_enc  = pd.get_dummies(X_tr,  columns=cat_cols, drop_first=False)
X_val_enc = pd.get_dummies(X_val, columns=cat_cols, drop_first=False)
X_test_enc= pd.get_dummies(X_test,columns=cat_cols, drop_first=False)

X_val_enc  = X_val_enc.reindex(columns=X_tr_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_tr_enc.columns, fill_value=0)

In [87]:
def sanitize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace(r"[\[\]<]", "_", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

def force_numeric(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype("category").cat.codes
    return df

X_tr_enc   = force_numeric(sanitize_columns(X_tr_enc))
X_val_enc  = force_numeric(sanitize_columns(X_val_enc))
X_test_enc = force_numeric(sanitize_columns(X_test_enc))

In [88]:
pos = y_tr.sum()
neg = len(y_tr) - pos
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 76.19839142091153


In [89]:
from xgboost import XGBClassifier

model_6m = XGBClassifier(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    eval_metric=["aucpr", "logloss"],
    random_state=42,
)

model_6m.fit(
    X_tr_enc, y_tr,
    verbose=50
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,"['aucpr', 'logloss']"


In [90]:
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np

y_prob_6m = model_6m.predict_proba(X_test_enc)[:, 1]

print("PR-AUC (6m):", average_precision_score(y_test, y_prob_6m))
print("ROC-AUC (6m):", roc_auc_score(y_test, y_prob_6m))

def recall_at_k(y_true, y_score, k_ratio=0.05):
    k = max(1, int(len(y_score) * k_ratio))
    idx = np.argsort(y_score)[::-1][:k]
    return y_true.iloc[idx].sum() / max(1, y_true.sum())

for k in [0.01, 0.03, 0.05, 0.10]:
    print(f"Recall@Top{k*100:.0f}% (6m):",
          recall_at_k(y_test, y_prob_6m, k))

PR-AUC (6m): 0.07595864166895794
ROC-AUC (6m): 0.8363332570492903
Recall@Top1% (6m): 0.1859504132231405
Recall@Top3% (6m): 0.3347107438016529
Recall@Top5% (6m): 0.43388429752066116
Recall@Top10% (6m): 0.5826446280991735


In [ ]:
# y_3m과 y_6m의 합계가 달라야 정상
print("y_test mean:", y_test.mean(), "sum:", y_test.sum())
print(train_df["y_3m"].sum(), train_df["y_6m"].sum(), test_df["y_3m"].sum(), test_df["y_6m"].sum())

y_test mean: 0.007803934214769429 sum: 242
746 1154 242 330


- 결론

본 연구에서는 도로 파손 발생이 극단적으로 희소한 문제 특성을 고려하여, 블록 단위 패널 데이터 기반 Top-K 우선점검 모델을 제안하였다. 

실험 결과, 다음 3개월 및 6개월 발생 여부를 각각 예측한 경우 모두에서 상위 5% 블록만 점검해도 전체 발생의 약 43%를 사전에 포착할 수 있음을 확인하였다. 

특히 예측 기간 확장에도 위험 블록의 상대적 순위가 안정적으로 유지되어, 제안 모델이 기간 설정에 강건한 공간적 위험 패턴을 학습하고 있음을 보여준다.

- 베이스라인 대비 약 10배 개선이긴 함 (기존 PR-AUC ≈ 0.0078, 새로운 PR-AUC ≈ 0.076으로 약 9.7배 개선)

- Top 5% 우선 점검 결과 해석

1) 전체 테스트 데이터 31,010건 중 모델 점수가 높은 **상위 5%(약 1,550건)**만을 우선 점검 대상으로 선정함

2) 실제 도로 파손 발생 건수는 총 242건으로, 전체 데이터 대비 약 0.78%의 극단적 희소 구조를 가짐

3) 상위 5% 점검 대상에 포함된 데이터 중 약 43%의 실제 파손 발생 건을 사전에 포함함 (Recall@Top5% = 0.434)

4) 이는 무작위로 5%를 선택할 경우 기대되는 파손 포착률(약 5%) 대비 약 8~9배 높은 성과. 즉, 제한된 점검 자원으로도 전체 파손의 절반에 가까운 구간을 선별적으로 식별 가능

5) 본 모델은 모든 구간을 정확히 분류하는 목적이 아닌, 우선 점검 대상의 효율적 랭킹(우선순위) 산정에 특화된 구조라고 볼 수 있으며 실제 운영 환경에서 유지보수 인력·예산 제약 하의 선제 점검 전략 수립에 활용 가능